Script_1_Carregamento

In [1]:
# ===================================================================
# GESTÃO DE DADOS DE PESQUISA SOCIOAMBIENTAL
# Baseado no Dicionário de Dados v2.4 e POP de Critérios de Qualidade
# ===================================================================

import pandas as pd
import numpy as np
import os
from datetime import datetime
import re

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("="*70)
print("PIPELINE DE GESTÃO DE DADOS - TEMPLE RESEARCH")
print("="*70)
print(f"Início da execução: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# ===================================================================
# 1. DEFINIÇÃO DOS CAMINHOS DOS ARQUIVOS
# ===================================================================

CAMINHOS = {
    'survey': '01_Base_Principal_Survey_v2_4.csv',
    'cadastro': '03_Cadastro_Mestre_Municipios_Comunidades.csv',
    'equipe': '04_Controle_Equipe_Campo.csv',
    'evidencias': '05_Controle_Evidencias.csv'
}

# Pasta para exportar resultados
PASTA_OUTPUT = 'output_tratado'
os.makedirs(PASTA_OUTPUT, exist_ok=True)

print("📁 Arquivos configurados:")
for nome, caminho in CAMINHOS.items():
    print(f"   - {nome}: {caminho}")
print()

# ===================================================================
# 2. FUNÇÃO PARA CARREGAR BASES COM TRATAMENTO DE ENCODING
# ===================================================================

def carregar_base(caminho, nome_base):
    """
    Tenta carregar CSV com diferentes encodings e separadores.
    """
    encodings = ['utf-8', 'utf-8-sig', 'latin1', 'iso-8859-1', 'cp1252']
    separadores = [',', ';', '\t']
    
    for encoding in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(caminho, sep=sep, encoding=encoding)
                print(f"✅ {nome_base}: Carregado com sucesso!")
                print(f"   - Linhas: {len(df)} | Colunas: {len(df.columns)}")
                print(f"   - Encoding: {encoding} | Separador: '{sep}'")
                return df
            except:
                continue
    
    raise ValueError(f"❌ Não foi possível carregar {nome_base}. Verifique o arquivo.")

# ===================================================================
# 3. CARREGAR TODAS AS BASES
# ===================================================================

print("🔄 Carregando bases de dados...")
print("-"*70)

bases = {}
for nome, caminho in CAMINHOS.items():
    try:
        bases[nome] = carregar_base(caminho, nome)
        print()
    except Exception as e:
        print(f"❌ Erro ao carregar {nome}: {e}")
        bases[nome] = None

# Verificar se todas foram carregadas
if any(df is None for df in bases.values()):
    print("\n⚠️ ALERTA: Algumas bases não foram carregadas. Verifique os arquivos.")
else:
    print("✅ Todas as bases carregadas com sucesso!")

PIPELINE DE GESTÃO DE DADOS - TEMPLE RESEARCH
Início da execução: 2026-07-23 03:58:13

📁 Arquivos configurados:
   - survey: 01_Base_Principal_Survey_v2_4.csv
   - cadastro: 03_Cadastro_Mestre_Municipios_Comunidades.csv
   - equipe: 04_Controle_Equipe_Campo.csv
   - evidencias: 05_Controle_Evidencias.csv

🔄 Carregando bases de dados...
----------------------------------------------------------------------
❌ Erro ao carregar survey: ❌ Não foi possível carregar survey. Verifique o arquivo.
❌ Erro ao carregar cadastro: ❌ Não foi possível carregar cadastro. Verifique o arquivo.
❌ Erro ao carregar equipe: ❌ Não foi possível carregar equipe. Verifique o arquivo.
❌ Erro ao carregar evidencias: ❌ Não foi possível carregar evidencias. Verifique o arquivo.

⚠️ ALERTA: Algumas bases não foram carregadas. Verifique os arquivos.


Script_2_Validacao_Estrutura

In [2]:
# ===================================================================
# VALIDAÇÃO DA ESTRUTURA DAS BASES
# ===================================================================

def validar_colunas_obrigatorias(df, colunas_esperadas, nome_base):
    """
    Verifica se todas as colunas esperadas estão presentes.
    """
    colunas_faltando = set(colunas_esperadas) - set(df.columns)
    colunas_extras = set(df.columns) - set(colunas_esperadas)
    
    if colunas_faltando:
        print(f"⚠️ {nome_base} - Colunas faltando: {colunas_faltando}")
    if colunas_extras:
        print(f"ℹ️ {nome_base} - Colunas extras: {colunas_extras}")
    
    return len(colunas_faltando) == 0

# Definição das colunas esperadas para cada base
COLUNAS_ESPERADAS = {
    'survey': [
        'ID', 'UUID', 'Versao_Form', 'Status', 'Municipio', 'Cod_IBGE',
        'Comunidade', 'Cod_Com', 'Supervisor', 'Pesquisador', 'Latitude',
        'Longitude', 'Foto', 'Consentimento', 'Genero', 'Idade',
        'Escolaridade', 'Renda', 'Q1_QV', 'Obs_Aberta'
    ],
    'cadastro': [
        'Cod_IBGE', 'Municipio_Padrao', 'Cod_Comunidade', 'Comunidade_Padrao',
        'Polo', 'Area', 'Status', 'Data_Atualizacao'
    ],
    'equipe': [
        'ID_Colaborador', 'Nome', 'Função', 'Polo', 'Bloco',
        'Municípios_Autorizados', 'Supervisor', 'Situação', 'Data_Início',
        'Data_Fim', 'Treinamento', 'Versão_Formulário_Habilitada', 'Observações'
    ],
    'evidencias': [
        'ID_Entrevista', 'UUID', 'Pesquisador', 'Município', 'Comunidade',
        'Termo_Consentimento', 'Arquivo_Termo', 'Foto_Entrevista',
        'Arquivo_Foto', 'Diario_Campo', 'Arquivo_Diario', 'Data_Coleta',
        'Data_Upload', 'Repositorio', 'Status_Validacao', 'Responsavel_QA',
        'Observacoes'
    ]
}

print("\n" + "="*70)
print("VALIDANDO ESTRUTURA DAS BASES")
print("="*70)

for nome, df in bases.items():
    if df is not None:
        print(f"\n📊 {nome}:")
        validar_colunas_obrigatorias(df, COLUNAS_ESPERADAS[nome], nome)
        print(f"   - Shape: {df.shape}")
        print(f"   - Tipos: {df.dtypes.value_counts().to_dict()}")


VALIDANDO ESTRUTURA DAS BASES


Script_3a_Diagnostico_Qualidade

In [3]:
# ===================================================================
# DIAGNÓSTICO DE QUALIDADE DOS DADOS
# ===================================================================
# Este script identifica e lista todos os problemas nas 4 (quatro) bases sem executar ações corretivas.
# Para cada problema, fornece:
#   a) Por que é um problema;
#   b) Qual seria o impacto;
#   c) Sugestão de preenchimento;
#   d) Os responsáveis: Supervisor, Pesquisador, Mobilizador ou UUID, Data e Local.
# ===================================================================

import pandas as pd
import numpy as np
import re
from datetime import datetime
from collections import defaultdict
import json
import warnings
warnings.filterwarnings('ignore')

# ===================================================================
# 1. FUNÇÕES AUXILIARES
# ===================================================================

def validar_equipe_referencia(df_equipe):
    print("\n" + "="*70)
    print("VALIDANDO BASE DE EQUIPE (REFERÊNCIA)")
    print("="*70)
    colunas_necessarias = ['Nome', 'Função', 'Supervisor', 'Situação']
    colunas_faltando = [col for col in colunas_necessarias if col not in df_equipe.columns]
    if colunas_faltando:
        print(f"⚠️ Colunas faltando: {colunas_faltando}")
        return None
    df_equipe_ativa = df_equipe[df_equipe['Situação'].str.upper().str.strip() == 'ATIVO'].copy()
    referencia = {
        'pesquisadores': {},
        'supervisores': {},
        'mobilizadores': {},
        'pesquisadores_por_supervisor': defaultdict(list),
        'supervisores_por_pesquisador': {},
        'todos_colaboradores': {}
    }
    for _, row in df_equipe_ativa.iterrows():
        nome = row['Nome']
        funcao = row['Função'].lower() if pd.notna(row['Função']) else ''
        supervisor = row['Supervisor'] if pd.notna(row['Supervisor']) else 'Não informado'
        referencia['todos_colaboradores'][nome] = {
            'funcao': row['Função'],
            'supervisor': supervisor,
            'polo': row.get('Polo', 'Não informado'),
            'bloco': row.get('Bloco', 'Não informado')
        }
        if 'pesquisador' in funcao or 'pesquisa' in funcao:
            referencia['pesquisadores'][nome] = {
                'supervisor': supervisor,
                'funcao': row['Função'],
                'polo': row.get('Polo', 'Não informado'),
                'bloco': row.get('Bloco', 'Não informado')
            }
            referencia['pesquisadores_por_supervisor'][supervisor].append(nome)
            referencia['supervisores_por_pesquisador'][nome] = supervisor
        elif 'mobilizador' in funcao or 'mobilização' in funcao:
            referencia['mobilizadores'][nome] = {
                'supervisor': supervisor,
                'funcao': row['Função'],
                'polo': row.get('Polo', 'Não informado'),
                'bloco': row.get('Bloco', 'Não informado')
            }
        elif 'supervisor' in funcao:
            referencia['supervisores'][nome] = {
                'funcao': row['Função'],
                'polo': row.get('Polo', 'Não informado'),
                'bloco': row.get('Bloco', 'Não informado')
            }
    print(f"✅ Equipe ativa: {len(df_equipe_ativa)} colaboradores")
    print(f"   - Pesquisadores: {len(referencia['pesquisadores'])}")
    print(f"   - Supervisores: {len(referencia['supervisores'])}")
    print(f"   - Mobilizadores: {len(referencia['mobilizadores'])}")
    return referencia

def validar_pesquisador_na_equipe(nome_pesquisador, referencia_equipe):
    if pd.isna(nome_pesquisador):
        return None, 'Pesquisador não informado', None
    nome_limpo = str(nome_pesquisador).strip()
    if nome_limpo in referencia_equipe['pesquisadores']:
        info = referencia_equipe['pesquisadores'][nome_limpo]
        return info['supervisor'], 'OK', info
    if nome_limpo in referencia_equipe['mobilizadores']:
        info = referencia_equipe['mobilizadores'][nome_limpo]
        return info['supervisor'], 'Mobilizador', info
    if nome_limpo in referencia_equipe['supervisores']:
        info = referencia_equipe['supervisores'][nome_limpo]
        return 'Supervisor', 'Supervisor', info
    return None, f'Não encontrado na equipe ativa', None

def obter_responsaveis(row, referencia_equipe):
    pesquisador = row.get('Pesquisador', 'Não informado')
    supervisor = row.get('Supervisor', 'Não informado')
    mobilizador = 'Não identificado'
    if pd.notna(pesquisador):
        supervisor_validado, status, info = validar_pesquisador_na_equipe(pesquisador, referencia_equipe)
        if supervisor_validado and supervisor_validado != 'Não informado':
            supervisor = supervisor_validado
        if status == 'Mobilizador':
            mobilizador = pesquisador
    if supervisor == 'Não informado' and pd.notna(pesquisador):
        if pesquisador in referencia_equipe['todos_colaboradores']:
            supervisor = referencia_equipe['todos_colaboradores'][pesquisador]['supervisor']
    return {
        'pesquisador': pesquisador,
        'supervisor': supervisor,
        'mobilizador': mobilizador
    }

# =========== FUNÇÃO DE SERIALIZAÇÃO CORRIGIDA ===========
def serializar_valor(valor):
    """
    Serializa valores para JSON, tratando arrays e diversos tipos.
    """
    # Se for array (numpy) ou lista, serializa recursivamente cada elemento
    if isinstance(valor, (list, tuple, np.ndarray)):
        return [serializar_valor(v) for v in valor]
    
    # Se for um valor nulo (NaN, None, NaT)
    if pd.isna(valor):
        return None
    
    # Timestamp -> string
    if isinstance(valor, pd.Timestamp):
        return str(valor)
    
    # Numéricos do numpy -> float nativo
    if isinstance(valor, (np.int64, np.float64)):
        return float(valor)
    
    # Dicionários -> recursão
    if isinstance(valor, dict):
        return {k: serializar_valor(v) for k, v in valor.items()}
    
    # Outros tipos (string, int, float, bool)
    return valor

# ===================================================================
# 2. DIAGNÓSTICO DE QUALIDADE
# ===================================================================

def diagnostico_qualidade(bases, referencia_equipe):
    print("\n" + "="*70)
    print("DIAGNÓSTICO DE QUALIDADE - IDENTIFICAÇÃO DE PROBLEMAS")
    print("="*70)
    todos_problemas = []
    resumo_por_tipo = defaultdict(int)
    resumo_por_base = defaultdict(int)

    # ----------------------------------------------------------------
    # 1. SURVEY
    # ----------------------------------------------------------------
    if 'survey' in bases and bases['survey'] is not None:
        df_survey = bases['survey']
        print("\n" + "="*70)
        print("1. DIAGNÓSTICO - BASE SURVEY")
        print("="*70)

        # 1.1 Pesquisadores não encontrados
        print("\n🔍 VALIDANDO PESQUISADORES NA EQUIPE")
        for idx, row in df_survey.iterrows():
            pesquisador = row.get('Pesquisador')
            if pd.notna(pesquisador):
                supervisor_validado, status, info = validar_pesquisador_na_equipe(pesquisador, referencia_equipe)
                if status != 'OK':
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'survey',
                        'tipo': 'pesquisador_nao_encontrado',
                        'ID': row.get('ID'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Municipio', 'Não informado'),
                        'a_porque': f"Pesquisador '{pesquisador}' não consta na equipe ativa.",
                        'b_impacto': "Impossível rastrear responsabilidade pela coleta. Dados podem ser de fonte não autorizada.",
                        'c_sugestao': "1. Verificar se o nome está correto; 2. Cadastrar o pesquisador na equipe; 3. Corrigir o nome.",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'status_validacao': status, 'supervisor_validado': supervisor_validado}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['pesquisador_nao_encontrado'] += 1
                    resumo_por_base['survey'] += 1

        # 1.2 Campos obrigatórios com nulos
        print("\n🔍 VALIDANDO CAMPOS OBRIGATÓRIOS")
        obrigatorios = ['ID', 'UUID', 'Versao_Form', 'Status', 'Municipio', 
                        'Cod_IBGE', 'Comunidade', 'Supervisor', 'Pesquisador',
                        'Consentimento', 'Genero', 'Idade', 'Escolaridade', 'Q1_QV']
        for col in obrigatorios:
            if col in df_survey.columns:
                registros_nulos = df_survey[df_survey[col].isna()]
                for idx, row in registros_nulos.iterrows():
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    descricoes = {
                        'Comunidade': "Campo 'Comunidade' está vazio. É obrigatório para identificação da localidade.",
                        'Genero': "Campo 'Gênero' está vazio. É obrigatório para análises demográficas.",
                        'Escolaridade': "Campo 'Escolaridade' está vazio. É obrigatório para estratificação da amostra.",
                        'Renda': "Campo 'Renda' está vazio. É obrigatório para análises socioeconômicas.",
                        'Idade': "Campo 'Idade' está vazio. É obrigatório para validação do consentimento.",
                        'Consentimento': "Campo 'Consentimento' está vazio. É obrigatório para conformidade ética.",
                        'Q1_QV': "Campo 'Q1_QV' está vazio. É obrigatório para análise de qualidade de vida.",
                        'Supervisor': "Campo 'Supervisor' está vazio. É obrigatório para rastreabilidade.",
                        'Pesquisador': "Campo 'Pesquisador' está vazio. É obrigatório para rastreabilidade.",
                        'Status': "Campo 'Status' está vazio. É obrigatório para controle do andamento."
                    }
                    impactos = {
                        'Comunidade': "Impossível georreferenciar ou analisar por comunidade. Perda de granularidade.",
                        'Genero': "Impossível fazer análises de gênero. Viés na amostra.",
                        'Escolaridade': "Impossível estratificar por nível educacional.",
                        'Renda': "Impossível análises de renda. Perda de informação socioeconômica.",
                        'Idade': "Não é possível validar elegibilidade do respondente.",
                        'Consentimento': "Risco ético. Não é possível confirmar consentimento.",
                        'Q1_QV': "Perda de indicador principal de qualidade de vida.",
                        'Supervisor': "Perda de rastreabilidade e responsabilidade.",
                        'Pesquisador': "Perda de rastreabilidade e responsabilidade.",
                        'Status': "Não é possível saber se entrevista está completa."
                    }
                    sugestoes = {
                        'Comunidade': "Preencher com a comunidade correta baseada no cadastro mestre.",
                        'Genero': "Consultar o respondente ou verificar no diário de campo.",
                        'Escolaridade': "Consultar o respondente ou verificar no diário de campo.",
                        'Renda': "Verificar se respondente não soube informar (registrar como 9999).",
                        'Idade': "Consultar o respondente ou verificar documento de identidade.",
                        'Consentimento': "Verificar se o termo foi assinado. Se não, registrar como 'Não'.",
                        'Q1_QV': "Consultar o respondente novamente ou verificar anotações de campo.",
                        'Supervisor': "Identificar supervisor baseado no pesquisador (usar 04_Controle_Equipe_Campo).",
                        'Pesquisador': "Verificar quem realizou a entrevista no diário de campo.",
                        'Status': "Verificar com o supervisor o status real da entrevista."
                    }
                    problema = {
                        'base': 'survey',
                        'tipo': 'campo_obrigatorio_nulo',
                        'campo': col,
                        'ID': row.get('ID'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Municipio', 'Não informado'),
                        'a_porque': descricoes.get(col, f"Campo '{col}' está vazio."),
                        'b_impacto': impactos.get(col, "Campo obrigatório não preenchido."),
                        'c_sugestao': sugestoes.get(col, "Preencher com informação correta ou justificar ausência."),
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'valor_atual': row.get(col), 'linha': idx}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['campo_obrigatorio_nulo'] += 1
                    resumo_por_base['survey'] += 1

        # 1.3 Idades inválidas
        print("\n🔍 VALIDANDO IDADES")
        if 'Idade' in df_survey.columns:
            registros_invalidos = df_survey[(df_survey['Idade'] < 18) | (df_survey['Idade'] > 120)]
            for idx, row in registros_invalidos.iterrows():
                responsaveis = obter_responsaveis(row, referencia_equipe)
                idade = row.get('Idade')
                problema = {
                    'base': 'survey',
                    'tipo': 'idade_invalida',
                    'ID': row.get('ID'),
                    'UUID': row.get('UUID'),
                    'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                    'Local': row.get('Municipio', 'Não informado'),
                    'a_porque': f"Idade '{idade}' está fora do intervalo permitido (18 a 120 anos).",
                    'b_impacto': "Respondente pode ser menor de idade (invalida consentimento) ou idade incorreta (viés).",
                    'c_sugestao': "1. Verificar com o pesquisador; 2. Se menor de 18, entrevista deve ser descartada; 3. Corrigir.",
                    'd_responsaveis': responsaveis,
                    'dados_adicionais': {'idade_atual': idade, 'linha': idx}
                }
                todos_problemas.append(problema)
                resumo_por_tipo['idade_invalida'] += 1
                resumo_por_base['survey'] += 1

        # 1.4 UUIDs duplicados
        print("\n🔍 VALIDANDO UUIDs ÚNICOS")
        if 'UUID' in df_survey.columns:
            df_duplicados = df_survey[df_survey['UUID'].duplicated(keep=False)]
            for uuid, group in df_duplicados.groupby('UUID'):
                registros = []
                for idx, row in group.iterrows():
                    resp = obter_responsaveis(row, referencia_equipe)
                    registros.append({
                        'ID': row.get('ID'),
                        'Pesquisador': resp['pesquisador'],
                        'Supervisor': resp['supervisor'],
                        'Data': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Municipio', 'Não informado')
                    })
                primeira_linha = group.iloc[0]
                responsaveis = obter_responsaveis(primeira_linha, referencia_equipe)
                problema = {
                    'base': 'survey',
                    'tipo': 'uuid_duplicado',
                    'ID': primeira_linha.get('ID'),
                    'UUID': uuid,
                    'Data_Coleta': primeira_linha.get('Data_Coleta', 'Não informado'),
                    'Local': primeira_linha.get('Municipio', 'Não informado'),
                    'a_porque': f"UUID '{uuid}' aparece {len(group)} vezes. Deve ser único por entrevista.",
                    'b_impacto': "Perda de rastreabilidade. Não é possível identificar unicamente cada entrevista.",
                    'c_sugestao': f"1. Gerar novos UUIDs para {len(group)-1} registros duplicados; 2. Manter apenas um UUID.",
                    'd_responsaveis': responsaveis,
                    'dados_adicionais': {'quantidade_duplicados': len(group), 'registros_afetados': registros}
                }
                todos_problemas.append(problema)
                resumo_por_tipo['uuid_duplicado'] += 1
                resumo_por_base['survey'] += 1

        # 1.5 IDs duplicados
        print("\n🔍 VALIDANDO IDs ÚNICOS")
        if 'ID' in df_survey.columns:
            df_duplicados = df_survey[df_survey['ID'].duplicated(keep=False)]
            for id_valor, group in df_duplicados.groupby('ID'):
                registros = []
                for idx, row in group.iterrows():
                    resp = obter_responsaveis(row, referencia_equipe)
                    registros.append({
                        'ID': row.get('ID'),
                        'UUID': row.get('UUID'),
                        'Pesquisador': resp['pesquisador'],
                        'Supervisor': resp['supervisor'],
                        'Data': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Municipio', 'Não informado')
                    })
                primeira_linha = group.iloc[0]
                responsaveis = obter_responsaveis(primeira_linha, referencia_equipe)
                problema = {
                    'base': 'survey',
                    'tipo': 'id_duplicado',
                    'ID': id_valor,
                    'UUID': primeira_linha.get('UUID'),
                    'Data_Coleta': primeira_linha.get('Data_Coleta', 'Não informado'),
                    'Local': primeira_linha.get('Municipio', 'Não informado'),
                    'a_porque': f"ID '{id_valor}' aparece {len(group)} vezes. Deve ser único por entrevista.",
                    'b_impacto': "Duplicação de registros. Pode causar superestimação nas análises.",
                    'c_sugestao': f"1. Identificar qual registro é o correto; 2. Remover os {len(group)-1} duplicados; 3. Manter o mais completo.",
                    'd_responsaveis': responsaveis,
                    'dados_adicionais': {'quantidade_duplicados': len(group), 'registros_afetados': registros}
                }
                todos_problemas.append(problema)
                resumo_por_tipo['id_duplicado'] += 1
                resumo_por_base['survey'] += 1

    # ----------------------------------------------------------------
    # 2. CADASTRO
    # ----------------------------------------------------------------
    if 'cadastro' in bases and bases['cadastro'] is not None:
        df_cadastro = bases['cadastro']
        print("\n" + "="*70)
        print("2. DIAGNÓSTICO - BASE CADASTRO")
        print("="*70)

        # 2.1 IBGE duplicado
        print("\n🔍 VALIDANDO CÓDIGOS IBGE")
        if 'Cod_IBGE' in df_cadastro.columns:
            ibge_duplicados = df_cadastro[df_cadastro['Cod_IBGE'].duplicated(keep=False)]
            if not ibge_duplicados.empty:
                for ibge, group in ibge_duplicados.groupby('Cod_IBGE'):
                    municipios = group['Municipio_Padrao'].tolist()
                    problema = {
                        'base': 'cadastro',
                        'tipo': 'ibge_duplicado',
                        'ID': None,
                        'UUID': None,
                        'Data_Coleta': None,
                        'Local': None,
                        'a_porque': f"Código IBGE '{ibge}' aparece {len(group)} vezes com diferentes municípios: {municipios}.",
                        'b_impacto': "Inconsistência no cadastro mestre. Dificulta padronização e cruzamento de dados.",
                        'c_sugestao': f"1. Verificar qual município é o correto; 2. Remover ou corrigir os incorretos; 3. Manter apenas um registro por código.",
                        'd_responsaveis': {'pesquisador': 'Cadastro Mestre', 'supervisor': 'Coordenação', 'mobilizador': 'Não aplicável'},
                        'dados_adicionais': {'cod_ibge': ibge, 'quantidade': len(group), 'municipios': municipios}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['ibge_duplicado'] += 1
                    resumo_por_base['cadastro'] += 1

        # 2.2 Status inválido
        print("\n🔍 VALIDANDO STATUS DOS MUNICÍPIOS")
        if 'Status' in df_cadastro.columns:
            status_invalidos = df_cadastro[~df_cadastro['Status'].str.upper().str.strip().isin(['ATIVO', 'INATIVO'])]
            if not status_invalidos.empty:
                for _, row in status_invalidos.iterrows():
                    problema = {
                        'base': 'cadastro',
                        'tipo': 'status_invalido',
                        'ID': None,
                        'UUID': None,
                        'Data_Coleta': None,
                        'Local': row.get('Municipio_Padrao'),
                        'a_porque': f"Status '{row.get('Status')}' para '{row.get('Municipio_Padrao')}' é inválido. Esperado: ATIVO/INATIVO.",
                        'b_impacto': "Não é possível saber se o município está ativo para coleta. Pode causar erros de planejamento.",
                        'c_sugestao': "1. Padronizar para ATIVO ou INATIVO; 2. Verificar com a coordenação.",
                        'd_responsaveis': {'pesquisador': 'Cadastro Mestre', 'supervisor': 'Coordenação', 'mobilizador': 'Não aplicável'},
                        'dados_adicionais': {'municipio': row.get('Municipio_Padrao'), 'status_atual': row.get('Status'), 'cod_ibge': row.get('Cod_IBGE')}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['status_invalido'] += 1
                    resumo_por_base['cadastro'] += 1

    # ----------------------------------------------------------------
    # 3. EQUIPE
    # ----------------------------------------------------------------
    if 'equipe' in bases and bases['equipe'] is not None:
        df_equipe = bases['equipe']
        print("\n" + "="*70)
        print("3. DIAGNÓSTICO - BASE EQUIPE")
        print("="*70)

        # 3.1 Supervisor inválido
        print("\n🔍 VALIDANDO RELAÇÃO SUPERVISOR-PESQUISADOR")
        if 'Supervisor' in df_equipe.columns and 'Nome' in df_equipe.columns:
            supervisores_validos = set(df_equipe[df_equipe['Função'].str.lower().str.contains('supervisor', na=False)]['Nome'])
            supervisores_invalidos = df_equipe[~df_equipe['Supervisor'].isin(supervisores_validos) & df_equipe['Supervisor'].notna()]
            if not supervisores_invalidos.empty:
                for _, row in supervisores_invalidos.iterrows():
                    problema = {
                        'base': 'equipe',
                        'tipo': 'supervisor_invalido',
                        'ID': None,
                        'UUID': None,
                        'Data_Coleta': None,
                        'Local': None,
                        'a_porque': f"Supervisor '{row.get('Supervisor')}' para '{row.get('Nome')}' não está cadastrado como supervisor.",
                        'b_impacto': "Hierarquia incorreta. Pode afetar a alocação de tarefas e responsabilidades.",
                        'c_sugestao': "1. Verificar se o supervisor está cadastrado; 2. Cadastrar ou corrigir o nome.",
                        'd_responsaveis': {'pesquisador': row.get('Nome'), 'supervisor': row.get('Supervisor'), 'mobilizador': 'Não aplicável'},
                        'dados_adicionais': {'colaborador': row.get('Nome'), 'funcao': row.get('Função'), 'supervisor_informado': row.get('Supervisor')}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['supervisor_invalido'] += 1
                    resumo_por_base['equipe'] += 1

        # 3.2 Situação inválida
        print("\n🔍 VALIDANDO SITUAÇÃO DOS COLABORADORES")
        if 'Situação' in df_equipe.columns:
            situacoes_validas = ['ATIVO', 'INATIVO', 'AFASTADO', 'FERIAS', 'DESLIGADO']
            situacoes_invalidas = df_equipe[~df_equipe['Situação'].str.upper().str.strip().isin(situacoes_validas)]
            if not situacoes_invalidas.empty:
                for _, row in situacoes_invalidas.iterrows():
                    problema = {
                        'base': 'equipe',
                        'tipo': 'situacao_invalida',
                        'ID': None,
                        'UUID': None,
                        'Data_Coleta': None,
                        'Local': None,
                        'a_porque': f"Situação '{row.get('Situação')}' para '{row.get('Nome')}' é inválida.",
                        'b_impacto': "Não é possível saber se o colaborador está disponível para trabalho.",
                        'c_sugestao': f"Padronizar para: {', '.join(situacoes_validas)}.",
                        'd_responsaveis': {'pesquisador': row.get('Nome'), 'supervisor': row.get('Supervisor'), 'mobilizador': 'Não aplicável'},
                        'dados_adicionais': {'colaborador': row.get('Nome'), 'funcao': row.get('Função'), 'situacao_atual': row.get('Situação')}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['situacao_invalida'] += 1
                    resumo_por_base['equipe'] += 1

        # 3.3 Versão do formulário inválida
        print("\n🔍 VALIDANDO VERSÃO DO FORMULÁRIO")
        if 'Versão_Formulário_Habilitada' in df_equipe.columns:
            versoes_invalidas = df_equipe[~df_equipe['Versão_Formulário_Habilitada'].str.upper().str.strip().isin(['V2.3', 'V2.4'])]
            if not versoes_invalidas.empty:
                for _, row in versoes_invalidas.iterrows():
                    problema = {
                        'base': 'equipe',
                        'tipo': 'versao_formulario_invalida',
                        'ID': None,
                        'UUID': None,
                        'Data_Coleta': None,
                        'Local': None,
                        'a_porque': f"Versão '{row.get('Versão_Formulário_Habilitada')}' para '{row.get('Nome')}' não é suportada.",
                        'b_impacto': "Pesquisador pode usar versão incorreta. Dados podem ser incompatíveis.",
                        'c_sugestao': "1. Atualizar para v2.4; 2. Verificar necessidade de manter versão antiga.",
                        'd_responsaveis': {'pesquisador': row.get('Nome'), 'supervisor': row.get('Supervisor'), 'mobilizador': 'Não aplicável'},
                        'dados_adicionais': {'colaborador': row.get('Nome'), 'versao_atual': row.get('Versão_Formulário_Habilitada')}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['versao_formulario_invalida'] += 1
                    resumo_por_base['equipe'] += 1

    # ----------------------------------------------------------------
    # 4. EVIDÊNCIAS
    # ----------------------------------------------------------------
    if 'evidencias' in bases and bases['evidencias'] is not None:
        df_evidencias = bases['evidencias']
        print("\n" + "="*70)
        print("4. DIAGNÓSTICO - BASE EVIDÊNCIAS")
        print("="*70)

        # 4.1 Pesquisadores não encontrados
        print("\n🔍 VALIDANDO PESQUISADORES NA EQUIPE")
        for idx, row in df_evidencias.iterrows():
            pesquisador = row.get('Pesquisador')
            if pd.notna(pesquisador):
                supervisor_validado, status, info = validar_pesquisador_na_equipe(pesquisador, referencia_equipe)
                if status != 'OK':
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'evidencias',
                        'tipo': 'pesquisador_nao_encontrado_evidencias',
                        'ID_Entrevista': row.get('ID_Entrevista'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Município', 'Não informado'),
                        'a_porque': f"Pesquisador '{pesquisador}' não consta na equipe ativa.",
                        'b_impacto': "Impossível rastrear responsabilidade pelas evidências.",
                        'c_sugestao': "1. Verificar nome; 2. Cadastrar na equipe; 3. Corrigir nome.",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'status_validacao': status, 'supervisor_validado': supervisor_validado}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['pesquisador_nao_encontrado_evidencias'] += 1
                    resumo_por_base['evidencias'] += 1

        # 4.2 Arquivos faltando
        print("\n🔍 VALIDANDO ARQUIVOS DE EVIDÊNCIAS")

        # Termo
        if 'Termo_Consentimento' in df_evidencias.columns and 'Arquivo_Termo' in df_evidencias.columns:
            tc_faltando = df_evidencias[(df_evidencias['Termo_Consentimento'] == 'Sim') & (df_evidencias['Arquivo_Termo'].isna())]
            if not tc_faltando.empty:
                for _, row in tc_faltando.iterrows():
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'evidencias',
                        'tipo': 'termo_consentimento_faltando',
                        'ID_Entrevista': row.get('ID_Entrevista'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Município', 'Não informado'),
                        'a_porque': f"Termo de consentimento faltando para ID {row.get('ID_Entrevista')}.",
                        'b_impacto': "Violação ética. Não é possível comprovar consentimento.",
                        'c_sugestao': "1. Solicitar ao pesquisador o envio; 2. Registrar justificativa.",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'arquivo_esperado': f"TC.{row.get('ID_Entrevista'):05d}.pdf"}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['termo_consentimento_faltando'] += 1
                    resumo_por_base['evidencias'] += 1

        # Foto
        if 'Foto_Entrevista' in df_evidencias.columns and 'Arquivo_Foto' in df_evidencias.columns:
            foto_faltando = df_evidencias[(df_evidencias['Foto_Entrevista'] == 'Sim') & (df_evidencias['Arquivo_Foto'].isna())]
            if not foto_faltando.empty:
                for _, row in foto_faltando.iterrows():
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'evidencias',
                        'tipo': 'foto_entrevista_faltando',
                        'ID_Entrevista': row.get('ID_Entrevista'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Município', 'Não informado'),
                        'a_porque': f"Foto da entrevista faltando para ID {row.get('ID_Entrevista')}.",
                        'b_impacto': "Perda de evidência visual. Dificulta validação.",
                        'c_sugestao': "1. Solicitar ao pesquisador o envio; 2. Registrar justificativa.",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'arquivo_esperado': f"IMG.{row.get('ID_Entrevista'):05d}.jpg"}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['foto_entrevista_faltando'] += 1
                    resumo_por_base['evidencias'] += 1

        # Diário
        if 'Arquivo_Diario' in df_evidencias.columns:
            diario_faltando = df_evidencias[df_evidencias['Arquivo_Diario'].isna()]
            if not diario_faltando.empty:
                for _, row in diario_faltando.iterrows():
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'evidencias',
                        'tipo': 'diario_campo_faltando',
                        'ID_Entrevista': row.get('ID_Entrevista'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Município', 'Não informado'),
                        'a_porque': f"Diário de campo faltando para ID {row.get('ID_Entrevista')}.",
                        'b_impacto': "Perda do registro detalhado. Não é possível auditar.",
                        'c_sugestao': "1. Solicitar ao pesquisador o envio; 2. Registrar justificativa.",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'arquivo_esperado': f"DC.{row.get('ID_Entrevista'):05d}.pdf"}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['diario_campo_faltando'] += 1
                    resumo_por_base['evidencias'] += 1

        # 4.3 Nomenclatura incorreta
        print("\n🔍 VALIDANDO NOMENCLATURA DOS ARQUIVOS (POP)")
        def validar_nomenclatura_pop(arquivo, prefixo):
            if pd.isna(arquivo):
                return True
            padrao = rf"^{prefixo}\.[0-9]{{5}}\.(pdf|jpg)$"
            return bool(re.match(padrao, str(arquivo), re.IGNORECASE))

        if 'Arquivo_Termo' in df_evidencias.columns:
            for idx, row in df_evidencias.iterrows():
                if pd.notna(row['Arquivo_Termo']) and not validar_nomenclatura_pop(row['Arquivo_Termo'], 'TC'):
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'evidencias',
                        'tipo': 'nomenclatura_incorreta',
                        'ID_Entrevista': row.get('ID_Entrevista'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Município', 'Não informado'),
                        'a_porque': f"Termo com nomenclatura incorreta: '{row['Arquivo_Termo']}'.",
                        'b_impacto': "Não segue padrão POP. Dificulta rastreabilidade.",
                        'c_sugestao': f"Renomear para: TC.{row.get('ID_Entrevista'):05d}.pdf",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'arquivo_atual': row['Arquivo_Termo'], 'arquivo_correto': f"TC.{row.get('ID_Entrevista'):05d}.pdf", 'tipo': 'Termo'}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['nomenclatura_incorreta'] += 1
                    resumo_por_base['evidencias'] += 1

        if 'Arquivo_Foto' in df_evidencias.columns:
            for idx, row in df_evidencias.iterrows():
                if pd.notna(row['Arquivo_Foto']) and not validar_nomenclatura_pop(row['Arquivo_Foto'], 'IMG'):
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'evidencias',
                        'tipo': 'nomenclatura_incorreta',
                        'ID_Entrevista': row.get('ID_Entrevista'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Município', 'Não informado'),
                        'a_porque': f"Foto com nomenclatura incorreta: '{row['Arquivo_Foto']}'.",
                        'b_impacto': "Não segue padrão POP. Dificulta rastreabilidade.",
                        'c_sugestao': f"Renomear para: IMG.{row.get('ID_Entrevista'):05d}.jpg",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'arquivo_atual': row['Arquivo_Foto'], 'arquivo_correto': f"IMG.{row.get('ID_Entrevista'):05d}.jpg", 'tipo': 'Foto'}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['nomenclatura_incorreta'] += 1
                    resumo_por_base['evidencias'] += 1

        if 'Arquivo_Diario' in df_evidencias.columns:
            for idx, row in df_evidencias.iterrows():
                if pd.notna(row['Arquivo_Diario']) and not validar_nomenclatura_pop(row['Arquivo_Diario'], 'DC'):
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'evidencias',
                        'tipo': 'nomenclatura_incorreta',
                        'ID_Entrevista': row.get('ID_Entrevista'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Município', 'Não informado'),
                        'a_porque': f"Diário com nomenclatura incorreta: '{row['Arquivo_Diario']}'.",
                        'b_impacto': "Não segue padrão POP. Dificulta rastreabilidade.",
                        'c_sugestao': f"Renomear para: DC.{row.get('ID_Entrevista'):05d}.pdf",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'arquivo_atual': row['Arquivo_Diario'], 'arquivo_correto': f"DC.{row.get('ID_Entrevista'):05d}.pdf", 'tipo': 'Diario'}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['nomenclatura_incorreta'] += 1
                    resumo_por_base['evidencias'] += 1

        # 4.4 Status de validação faltando
        print("\n🔍 VALIDANDO STATUS DE VALIDAÇÃO")
        if 'Status_Validacao' in df_evidencias.columns:
            status_faltando = df_evidencias[df_evidencias['Status_Validacao'].isna()]
            if not status_faltando.empty:
                for _, row in status_faltando.iterrows():
                    responsaveis = obter_responsaveis(row, referencia_equipe)
                    problema = {
                        'base': 'evidencias',
                        'tipo': 'status_validacao_faltando',
                        'ID_Entrevista': row.get('ID_Entrevista'),
                        'UUID': row.get('UUID'),
                        'Data_Coleta': row.get('Data_Coleta', 'Não informado'),
                        'Local': row.get('Município', 'Não informado'),
                        'a_porque': f"Status de validação não preenchido para ID {row.get('ID_Entrevista')}.",
                        'b_impacto': "Não é possível saber se a evidência foi validada.",
                        'c_sugestao': "Preencher com APROVADO, REPROVADO ou PENDENTE.",
                        'd_responsaveis': responsaveis,
                        'dados_adicionais': {'status_atual': row.get('Status_Validacao'), 'responsavel_qa': row.get('Responsavel_QA', 'Não informado')}
                    }
                    todos_problemas.append(problema)
                    resumo_por_tipo['status_validacao_faltando'] += 1
                    resumo_por_base['evidencias'] += 1

        # 4.5 Datas inconsistentes
        print("\n🔍 VALIDANDO DATAS DE COLETA E UPLOAD")
        try:
            if 'Data_Coleta' in df_evidencias.columns and 'Data_Upload' in df_evidencias.columns:
                df_evidencias['Data_Coleta'] = pd.to_datetime(df_evidencias['Data_Coleta'], errors='coerce')
                df_evidencias['Data_Upload'] = pd.to_datetime(df_evidencias['Data_Upload'], errors='coerce')
                datas_inconsistentes = df_evidencias[
                    (df_evidencias['Data_Upload'].notna()) & 
                    (df_evidencias['Data_Coleta'].notna()) & 
                    (df_evidencias['Data_Upload'] < df_evidencias['Data_Coleta'])
                ]
                if not datas_inconsistentes.empty:
                    for _, row in datas_inconsistentes.iterrows():
                        responsaveis = obter_responsaveis(row, referencia_equipe)
                        problema = {
                            'base': 'evidencias',
                            'tipo': 'data_inconsistente',
                            'ID_Entrevista': row.get('ID_Entrevista'),
                            'UUID': row.get('UUID'),
                            'Data_Coleta': row.get('Data_Coleta'),
                            'Local': row.get('Município', 'Não informado'),
                            'a_porque': f"Upload ({row.get('Data_Upload')}) anterior à coleta ({row.get('Data_Coleta')}).",
                            'b_impacto': "Inconsistência temporal. Indica possível erro ou fraude.",
                            'c_sugestao': "1. Verificar datas com o pesquisador; 2. Corrigir a data incorreta; 3. Registrar justificativa.",
                            'd_responsaveis': responsaveis,
                            'dados_adicionais': {'data_coleta': str(row.get('Data_Coleta')), 'data_upload': str(row.get('Data_Upload')), 'diferenca_dias': (row.get('Data_Upload') - row.get('Data_Coleta')).days}
                        }
                        todos_problemas.append(problema)
                        resumo_por_tipo['data_inconsistente'] += 1
                        resumo_por_base['evidencias'] += 1
        except Exception as e:
            print(f"⚠️ Erro ao validar datas: {e}")

    # ===================================================================
    # RESUMO
    # ===================================================================
    print("\n" + "="*70)
    print("RESUMO DO DIAGNÓSTICO DE QUALIDADE")
    print("="*70)
    print(f"\n📊 TOTAL DE PROBLEMAS IDENTIFICADOS: {len(todos_problemas)}")
    print("\n📋 DISTRIBUIÇÃO POR BASE:")
    for base, qtd in sorted(resumo_por_base.items(), key=lambda x: x[1], reverse=True):
        print(f"   - {base}: {qtd}")
    print("\n📋 DISTRIBUIÇÃO POR TIPO:")
    for tipo, qtd in sorted(resumo_por_tipo.items(), key=lambda x: x[1], reverse=True):
        print(f"   - {tipo}: {qtd}")

    return todos_problemas, resumo_por_tipo, resumo_por_base

# ===================================================================
# FUNÇÃO PRINCIPAL - EXECUTAR DIAGNÓSTICO
# ===================================================================

def executar_diagnostico(bases):
    print("\n" + "="*70)
    print("INICIANDO DIAGNÓSTICO DE QUALIDADE")
    print("="*70)

    if 'equipe' in bases and bases['equipe'] is not None:
        referencia_equipe = validar_equipe_referencia(bases['equipe'])
        if referencia_equipe is None:
            print("❌ Erro: Não foi possível validar a base de equipe.")
            return None
    else:
        print("❌ Erro: Base de equipe não encontrada.")
        return None

    problemas, _, _ = diagnostico_qualidade(bases, referencia_equipe)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    arquivo_json = f"diagnostico_qualidade_{timestamp}.json"
    problemas_serializaveis = []
    for p in problemas:
        p_serial = {}
        for key, value in p.items():
            p_serial[key] = serializar_valor(value)
        problemas_serializaveis.append(p_serial)

    with open(arquivo_json, 'w', encoding='utf-8') as f:
        json.dump(problemas_serializaveis, f, ensure_ascii=False, indent=2)
    print(f"\n📄 Diagnóstico salvo em: {arquivo_json}")
    return problemas

# ===================================================================
# EXECUÇÃO
# ===================================================================

try:
    if 'bases' in globals() and bases is not None:
        resultados = executar_diagnostico(bases)
    else:
        print("⚠️ Bases não carregadas. Execute o Script 1 primeiro.")
except NameError:
    print("⚠️ Variável 'bases' não definida. Execute o Script 1 primeiro.")
except Exception as e:
    print(f"❌ Erro: {e}")
    import traceback
    traceback.print_exc()


INICIANDO DIAGNÓSTICO DE QUALIDADE
❌ Erro: Base de equipe não encontrada.


Script_3b_Decisoes_Acoes

In [4]:
# ===================================================================
# DECISÕES E AÇÕES DE PRIORIZAÇÃO E A RECOMENDAÇÃO
# ===================================================================
# Este script recebe o diagnóstico e gera:
#   - Priorização dos problemas (Crítico, Alto, Médio, Baixo)
#   - Recomendações específicas para cada tipo de problema
#   - Plano de ação consolidado com responsáveis
#   - Sugestões de correção automática, se for possível aplicar
# ===================================================================

import pandas as pd
import numpy as np
import json
from datetime import datetime
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ===================================================================
# 1. DEFINIÇÃO DE PRIORIDADES E AÇÕES
# ===================================================================

PRIORIDADES = {
    'critico': [
        'idade_invalida',
        'termo_consentimento_faltando',
        'diario_campo_faltando',
        'uuid_duplicado',
        'id_duplicado',
        'data_inconsistente'
    ],
    'alto': [
        'pesquisador_nao_encontrado',
        'pesquisador_nao_encontrado_evidencias',
        'campo_obrigatorio_nulo',
        'foto_entrevista_faltando'
    ],
    'medio': [
        'nomenclatura_incorreta',
        'status_validacao_faltando',
        'versao_formulario_invalida'
    ],
    'baixo': [
        'status_invalido',
        'supervisor_invalido',
        'situacao_invalida',
        'ibge_duplicado'
    ]
}

# Ações específicas por tipo
ACOES = {
    'pesquisador_nao_encontrado': {
        'descricao': 'Pesquisador não cadastrado na equipe',
        'acao_automatica': False,
        'acao_manual': 'Verificar nome e cadastrar na equipe ou corrigir o nome no registro',
        'responsavel': 'Coordenação de Campo',
        'prazo_sugerido': 'Imediato'
    },
    'pesquisador_nao_encontrado_evidencias': {
        'descricao': 'Pesquisador das evidências não cadastrado na equipe',
        'acao_automatica': False,
        'acao_manual': 'Verificar nome e cadastrar na equipe ou corrigir o nome no registro',
        'responsavel': 'Coordenação de Campo',
        'prazo_sugerido': 'Imediato'
    },
    'campo_obrigatorio_nulo': {
        'descricao': 'Campo obrigatório não preenchido',
        'acao_automatica': True,
        'acao_manual': 'Preencher com dados do cadastro mestre ou consultar respondente',
        'responsavel': 'Pesquisador responsável',
        'prazo_sugerido': 'Até 3 dias'
    },
    'idade_invalida': {
        'descricao': 'Idade fora do intervalo permitido (18-120)',
        'acao_automatica': False,
        'acao_manual': 'Verificar com o respondente ou descartar entrevista se menor de 18',
        'responsavel': 'Supervisor do pesquisador',
        'prazo_sugerido': 'Imediato'
    },
    'uuid_duplicado': {
        'descricao': 'UUID duplicado entre registros',
        'acao_automatica': True,
        'acao_manual': 'Gerar novos UUIDs para os duplicados',
        'responsavel': 'Analista de Dados',
        'prazo_sugerido': 'Até 1 dia'
    },
    'id_duplicado': {
        'descricao': 'ID duplicado entre registros',
        'acao_automatica': True,
        'acao_manual': 'Remover duplicatas mantendo o registro mais completo',
        'responsavel': 'Analista de Dados',
        'prazo_sugerido': 'Até 1 dia'
    },
    'nomenclatura_incorreta': {
        'descricao': 'Arquivo com nomenclatura fora do padrão POP',
        'acao_automatica': True,
        'acao_manual': 'Renomear arquivos seguindo o padrão TC/IMG/DC.#####.ext',
        'responsavel': 'Analista de Dados',
        'prazo_sugerido': 'Até 2 dias'
    },
    'termo_consentimento_faltando': {
        'descricao': 'Termo de consentimento não enviado',
        'acao_automatica': False,
        'acao_manual': 'Solicitar ao pesquisador o envio do termo assinado',
        'responsavel': 'Supervisor do pesquisador',
        'prazo_sugerido': 'Imediato'
    },
    'foto_entrevista_faltando': {
        'descricao': 'Foto da entrevista não enviada',
        'acao_automatica': False,
        'acao_manual': 'Solicitar ao pesquisador o envio da foto',
        'responsavel': 'Supervisor do pesquisador',
        'prazo_sugerido': 'Até 1 dia'
    },
    'diario_campo_faltando': {
        'descricao': 'Diário de campo não enviado',
        'acao_automatica': False,
        'acao_manual': 'Solicitar ao pesquisador o envio do diário',
        'responsavel': 'Supervisor do pesquisador',
        'prazo_sugerido': 'Imediato'
    },
    'data_inconsistente': {
        'descricao': 'Data de upload anterior à data de coleta',
        'acao_automatica': False,
        'acao_manual': 'Verificar datas corretas com o pesquisador',
        'responsavel': 'Supervisor do pesquisador',
        'prazo_sugerido': 'Até 1 dia'
    },
    'status_invalido': {
        'descricao': 'Status do município inválido (esperado ATIVO/INATIVO)',
        'acao_automatica': True,
        'acao_manual': 'Padronizar status para ATIVO ou INATIVO',
        'responsavel': 'Coordenação',
        'prazo_sugerido': 'Até 3 dias'
    },
    'ibge_duplicado': {
        'descricao': 'Código IBGE duplicado com municípios diferentes',
        'acao_automatica': False,
        'acao_manual': 'Verificar qual município é correto e remover duplicatas',
        'responsavel': 'Coordenação',
        'prazo_sugerido': 'Até 5 dias'
    },
    'supervisor_invalido': {
        'descricao': 'Supervisor não cadastrado como supervisor',
        'acao_automatica': False,
        'acao_manual': 'Cadastrar supervisor ou corrigir nome na equipe',
        'responsavel': 'Coordenação de Campo',
        'prazo_sugerido': 'Até 2 dias'
    },
    'situacao_invalida': {
        'descricao': 'Situação do colaborador inválida',
        'acao_automatica': True,
        'acao_manual': f"Padronizar para: ATIVO, INATIVO, AFASTADO, FERIAS, DESLIGADO",
        'responsavel': 'Coordenação de Campo',
        'prazo_sugerido': 'Até 2 dias'
    },
    'versao_formulario_invalida': {
        'descricao': 'Versão do formulário não suportada',
        'acao_automatica': True,
        'acao_manual': 'Atualizar para v2.4 (versão mais recente)',
        'responsavel': 'Coordenação de Campo',
        'prazo_sugerido': 'Até 2 dias'
    },
    'status_validacao_faltando': {
        'descricao': 'Status de validação não preenchido',
        'acao_automatica': False,
        'acao_manual': 'Preencher com APROVADO, REPROVADO ou PENDENTE',
        'responsavel': 'QA Responsável',
        'prazo_sugerido': 'Até 2 dias'
    }
}

# ===================================================================
# 2. FUNÇÕES DE ANÁLISE E DECISÃO
# ===================================================================

def classificar_prioridade(tipo_problema):
    """
    Classifica a prioridade de um problema com base no tipo
    """
    for prioridade, tipos in PRIORIDADES.items():
        if tipo_problema in tipos:
            return prioridade
    return 'baixo'  # default

def gerar_recomendacoes(problemas):
    """
    Gera recomendações consolidadas a partir da lista de problemas
    """
    # Agrupar por tipo
    por_tipo = defaultdict(list)
    for p in problemas:
        por_tipo[p['tipo']].append(p)
    
    recomendacoes = []
    
    for tipo, lista in por_tipo.items():
        prioridade = classificar_prioridade(tipo)
        acao = ACOES.get(tipo, {
            'descricao': 'Problema não classificado',
            'acao_automatica': False,
            'acao_manual': 'Investigar manualmente',
            'responsavel': 'Analista',
            'prazo_sugerido': 'A definir'
        })
        
        # Coletar IDs afetados
        ids_afetados = []
        for p in lista:
            if 'ID' in p and p['ID']:
                ids_afetados.append(str(p['ID']))
            elif 'ID_Entrevista' in p and p['ID_Entrevista']:
                ids_afetados.append(str(p['ID_Entrevista']))
        
        # Coletar responsáveis
        responsaveis = set()
        for p in lista:
            if 'd_responsaveis' in p:
                resp = p['d_responsaveis']
                if resp.get('pesquisador') and resp['pesquisador'] != 'Não informado':
                    responsaveis.add(f"Pesquisador: {resp['pesquisador']}")
                if resp.get('supervisor') and resp['supervisor'] != 'Não informado':
                    responsaveis.add(f"Supervisor: {resp['supervisor']}")
                if resp.get('mobilizador') and resp['mobilizador'] != 'Não identificado':
                    responsaveis.add(f"Mobilizador: {resp['mobilizador']}")
        
        # Estatísticas
        qtd_bases = len(set(p.get('base', '') for p in lista))
        
        recomendacao = {
            'tipo': tipo,
            'quantidade': len(lista),
            'prioridade': prioridade.upper(),
            'descricao': acao['descricao'],
            'acao_automatica': acao['acao_automatica'],
            'acao_manual': acao['acao_manual'],
            'responsavel': acao['responsavel'],
            'prazo_sugerido': acao['prazo_sugerido'],
            'bases_afetadas': qtd_bases,
            'ids_afetados': ids_afetados[:10],  # limitar a 10
            'total_ids': len(ids_afetados),
            'responsaveis_envolvidos': list(responsaveis)[:5]  # limitar a 5
        }
        recomendacoes.append(recomendacao)
    
    # Ordenar por prioridade
    ordem_prioridade = {'CRITICO': 0, 'ALTO': 1, 'MEDIO': 2, 'BAIXO': 3}
    recomendacoes.sort(key=lambda x: ordem_prioridade.get(x['prioridade'], 4))
    
    return recomendacoes

def gerar_plano_acao(recomendacoes):
    """
    Gera um plano de ação consolidado a partir das recomendações
    """
    plano = {
        'data_geracao': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'total_problemas': sum(r['quantidade'] for r in recomendacoes),
        'prioridades': defaultdict(int),
        'acoes_automaticas': [],
        'acoes_manuais': [],
        'responsaveis': defaultdict(list),
        'cronograma': defaultdict(list)
    }
    
    for rec in recomendacoes:
        prioridade = rec['prioridade'].lower()
        plano['prioridades'][prioridade] += rec['quantidade']
        
        if rec['acao_automatica']:
            plano['acoes_automaticas'].append({
                'tipo': rec['tipo'],
                'quantidade': rec['quantidade'],
                'acao': rec['acao_manual']
            })
        else:
            plano['acoes_manuais'].append({
                'tipo': rec['tipo'],
                'quantidade': rec['quantidade'],
                'acao': rec['acao_manual'],
                'responsavel': rec['responsavel']
            })
        
        # Responsáveis
        plano['responsaveis'][rec['responsavel']].append({
            'tipo': rec['tipo'],
            'quantidade': rec['quantidade']
        })
        
        # Cronograma
        plano['cronograma'][rec['prazo_sugerido']].append({
            'tipo': rec['tipo'],
            'quantidade': rec['quantidade']
        })
    
    return plano

# ===================================================================
# 3. FUNÇÃO PRINCIPAL - DECISÕES E AÇÕES
# ===================================================================

def executar_acoes(problemas, arquivo_saida=None):
    """
    Executa a análise de decisões e ações a partir do diagnóstico
    """
    print("\n" + "="*70)
    print("DECISÕES E AÇÕES - PRIORIZAÇÃO E RECOMENDAÇÕES")
    print("="*70)
    
    if not problemas:
        print("\n✅ Nenhum problema identificado. Nenhuma ação necessária.")
        return None
    
    print(f"\n📊 Processando {len(problemas)} problemas identificados...")
    
    # Gerar recomendações
    recomendacoes = gerar_recomendacoes(problemas)
    
    # Gerar plano de ação
    plano = gerar_plano_acao(recomendacoes)
    
    # Exibir resumo
    print("\n" + "="*70)
    print("RESUMO DE DECISÕES E AÇÕES")
    print("="*70)
    
    print(f"\n📊 TOTAL DE PROBLEMAS: {plano['total_problemas']}")
    
    print("\n🎯 DISTRIBUIÇÃO POR PRIORIDADE:")
    for prioridade in ['critico', 'alto', 'medio', 'baixo']:
        qtd = plano['prioridades'].get(prioridade, 0)
        if qtd > 0:
            print(f"   {prioridade.upper()}: {qtd}")
            if prioridade == 'critico':
                print("      ⚠️ Resolver IMEDIATAMENTE")
    
    print("\n📋 AÇÕES AUTOMÁTICAS POSSÍVEIS:")
    if plano['acoes_automaticas']:
        for acao in plano['acoes_automaticas']:
            print(f"   - {acao['tipo']}: {acao['quantidade']} ocorrência(s)")
            print(f"     Ação: {acao['acao']}")
    else:
        print("   Nenhuma ação automática identificada.")
    
    print("\n📋 AÇÕES MANUAIS NECESSÁRIAS:")
    if plano['acoes_manuais']:
        # Agrupar por responsável
        por_responsavel = defaultdict(list)
        for acao in plano['acoes_manuais']:
            por_responsavel[acao['responsavel']].append(acao)
        
        for responsavel, acoes in por_responsavel.items():
            print(f"\n   {responsavel}:")
            for acao in acoes:
                print(f"     - {acao['tipo']}: {acao['quantidade']} ocorrência(s)")
                print(f"       Ação: {acao['acao']}")
    else:
        print("   Nenhuma ação manual necessária.")
    
    print("\n📅 CRONOGRAMA SUGERIDO:")
    for prazo in ['Imediato', 'Até 1 dia', 'Até 2 dias', 'Até 3 dias', 'Até 5 dias']:
        if prazo in plano['cronograma']:
            itens = plano['cronograma'][prazo]
            total = sum(item['quantidade'] for item in itens)
            print(f"   {prazo}: {total} ocorrência(s)")
            for item in itens:
                print(f"     - {item['tipo']}: {item['quantidade']}")
    
    # Salvar resultados
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    if arquivo_saida is None:
        arquivo_saida = f"decisoes_acoes_{timestamp}.json"
    
    # Converter para serializável
    plano_serializavel = {
        'data_geracao': plano['data_geracao'],
        'total_problemas': plano['total_problemas'],
        'prioridades': dict(plano['prioridades']),
        'acoes_automaticas': plano['acoes_automaticas'],
        'acoes_manuais': plano['acoes_manuais'],
        'responsaveis': {k: list(v) for k, v in plano['responsaveis'].items()},
        'cronograma': {k: list(v) for k, v in plano['cronograma'].items()}
    }
    
    with open(arquivo_saida, 'w', encoding='utf-8') as f:
        json.dump(plano_serializavel, f, ensure_ascii=False, indent=2)
    
    print(f"\n📄 Plano de ação salvo em: {arquivo_saida}")
    
    # Salvar também em Markdown para relatório executivo
    md_file = f"plano_acao_{timestamp}.md"
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write(f"# PLANO DE AÇÃO - CORREÇÃO DE PROBLEMAS\n\n")
        f.write(f"**Data de geração:** {plano['data_geracao']}\n\n")
        f.write(f"**Total de problemas:** {plano['total_problemas']}\n\n")
        
        f.write("## Priorização\n\n")
        f.write("| Prioridade | Quantidade | Ação |\n")
        f.write("|------------|------------|------|\n")
        for prioridade in ['critico', 'alto', 'medio', 'baixo']:
            qtd = plano['prioridades'].get(prioridade, 0)
            if qtd > 0:
                acao = "⚠️ IMEDIATO" if prioridade == 'critico' else "Prioritário" if prioridade == 'alto' else "Agendar"
                f.write(f"| {prioridade.upper()} | {qtd} | {acao} |\n")
        
        f.write("\n## Ações por Responsável\n\n")
        for responsavel, acoes in plano['responsaveis'].items():
            f.write(f"### {responsavel}\n")
            for acao in acoes:
                f.write(f"- **{acao['tipo']}**: {acao['quantidade']} ocorrência(s)\n")
            f.write("\n")
        
        f.write("\n## Cronograma\n\n")
        for prazo in ['Imediato', 'Até 1 dia', 'Até 2 dias', 'Até 3 dias', 'Até 5 dias']:
            if prazo in plano['cronograma']:
                itens = plano['cronograma'][prazo]
                total = sum(item['quantidade'] for item in itens)
                f.write(f"### {prazo} ({total} ocorrências)\n")
                for item in itens:
                    f.write(f"- {item['tipo']}: {item['quantidade']}\n")
                f.write("\n")
    
    print(f"📄 Relatório executivo salvo em: {md_file}")
    
    return plano

# ===================================================================
# 4. EXECUÇÃO - CARREGAR DIAGNÓSTICO OU USAR DIRETAMENTE
# ===================================================================

# Verificar se o diagnóstico já foi executado e há problemas
try:
    if 'resultados' in globals() and resultados is not None:
        # Executar ações com base no diagnóstico
        plano = executar_acoes(resultados)
    else:
        # Tentar carregar diagnóstico de arquivo
        import glob
        arquivos_diagnostico = glob.glob('diagnostico_qualidade_*.json')
        if arquivos_diagnostico:
            ultimo_arquivo = sorted(arquivos_diagnostico)[-1]
            print(f"📂 Carregando diagnóstico de: {ultimo_arquivo}")
            with open(ultimo_arquivo, 'r', encoding='utf-8') as f:
                problemas = json.load(f)
            plano = executar_acoes(problemas, arquivo_saida='decisoes_acoes_carregado.json')
        else:
            print("⚠️ Nenhum diagnóstico encontrado. Execute o Script 3a primeiro.")
except NameError:
    print("⚠️ Variável 'resultados' não definida.")
    print("   Execute o Script 3a primeiro ou carregue um arquivo de diagnóstico.")
except Exception as e:
    print(f"❌ Erro durante a execução: {e}")
    import traceback
    traceback.print_exc()

⚠️ Nenhum diagnóstico encontrado. Execute o Script 3a primeiro.


Script_4_Tratamento_Padronizacao

In [5]:
# ===================================================================
# TRATAMENTO E PADRONIZAÇÃO DOS DADOS
# ===================================================================

import pandas as pd
import numpy as np
import re
from datetime import datetime
import os

print("="*70)
print("INICIANDO TRATAMENTO E PADRONIZAÇÃO DOS DADOS")
print("="*70)

# ===================================================================
# 1. FUNÇÕES DE PADRONIZAÇÃO
# ===================================================================

def padronizar_cadastro(df):
    """
    Padroniza a base de cadastro mestre
    """
    df_pad = df.copy()
    if 'Cod_IBGE' in df_pad.columns:
        df_pad['Cod_IBGE'] = df_pad['Cod_IBGE'].astype(str).str.zfill(7)
    if 'Status' in df_pad.columns:
        df_pad['Status'] = df_pad['Status'].str.upper().str.strip()
    return df_pad

def padronizar_equipe(df):
    """
    Padroniza a base de controle de equipe
    """
    df_pad = df.copy()
    if 'Situação' in df_pad.columns:
        df_pad['Situação'] = df_pad['Situação'].str.upper().str.strip()
    if 'Função' in df_pad.columns:
        df_pad['Função'] = df_pad['Função'].str.title().str.strip()
    colunas_data = ['Data_Início', 'Data_Fim']
    for col in colunas_data:
        if col in df_pad.columns:
            try:
                df_pad[col] = pd.to_datetime(df_pad[col])
            except:
                pass
    return df_pad

def padronizar_survey(df, cadastro_df):
    """
    Padroniza a base de survey com as novas regras de qualidade
    """
    df_pad = df.copy()
    df_pad['Qualidade_Registro'] = 'OK'
    
    # --- 1. IDs duplicados (NÃO REMOVER, apenas marcar) ---
    if 'ID' in df_pad.columns:
        ids_duplicados = df_pad[df_pad['ID'].duplicated(keep=False)]
        if not ids_duplicados.empty:
            df_pad.loc[df_pad['ID'].isin(ids_duplicados['ID'].unique()), 'Qualidade_Registro'] = 'ID_Duplicado'
            print(f"   ⚠️ IDs duplicados marcados: {len(ids_duplicados)} registros")
    
    # --- 2. UUIDs duplicados (marcar, mas não remover) ---
    if 'UUID' in df_pad.columns:
        uuids_duplicados = df_pad[df_pad['UUID'].duplicated(keep=False)]
        if not uuids_duplicados.empty:
            # Se já tiver marcação, concatena
            mask = df_pad['UUID'].isin(uuids_duplicados['UUID'].unique())
            df_pad.loc[mask, 'Qualidade_Registro'] = df_pad.loc[mask, 'Qualidade_Registro'].apply(
                lambda x: 'UUID_Duplicado' if x == 'OK' else x + '|UUID_Duplicado'
            )
            print(f"   ⚠️ UUIDs duplicados marcados: {len(uuids_duplicados)} registros")
    
    # --- 3. Gênero inválido (padronizar E marcar) ---
    if 'Genero' in df_pad.columns:
        df_pad['Genero'] = df_pad['Genero'].str.upper().str.strip()
        mask_genero_invalido = ~df_pad['Genero'].isin(['M', 'F']) & df_pad['Genero'].notna()
        if mask_genero_invalido.any():
            df_pad.loc[mask_genero_invalido, 'Genero'] = np.nan
            df_pad.loc[mask_genero_invalido, 'Qualidade_Registro'] = df_pad.loc[mask_genero_invalido, 'Qualidade_Registro'].apply(
                lambda x: 'Genero_Invalido' if x == 'OK' else x + '|Genero_Invalido'
            )
            print(f"   ⚠️ Gêneros inválidos padronizados e marcados: {mask_genero_invalido.sum()} registros")
    
    # --- 4. Q1_QV fora da escala (NÃO CORRIGIR, apenas marcar) ---
    if 'Q1_QV' in df_pad.columns:
        mask_q1_invalido = ~df_pad['Q1_QV'].isin([1, 2, 3, 4, 5]) & df_pad['Q1_QV'].notna()
        if mask_q1_invalido.any():
            df_pad.loc[mask_q1_invalido, 'Qualidade_Registro'] = df_pad.loc[mask_q1_invalido, 'Qualidade_Registro'].apply(
                lambda x: 'Q1_QV_Invalido' if x == 'OK' else x + '|Q1_QV_Invalido'
            )
            print(f"   ⚠️ Q1_QV fora da escala marcados: {mask_q1_invalido.sum()} registros (valores mantidos)")
    
    # --- 5. Renda negativa (NÃO CORRIGIR, apenas marcar) ---
    if 'Renda' in df_pad.columns:
        mask_renda_neg = df_pad['Renda'] < 0
        if mask_renda_neg.any():
            df_pad.loc[mask_renda_neg, 'Qualidade_Registro'] = df_pad.loc[mask_renda_neg, 'Qualidade_Registro'].apply(
                lambda x: 'Renda_Negativa' if x == 'OK' else x + '|Renda_Negativa'
            )
            print(f"   ⚠️ Rendas negativas marcadas: {mask_renda_neg.sum()} registros (valores mantidos)")
    
    # --- 6. Idade inválida (corrigir para NaN e marcar) ---
    if 'Idade' in df_pad.columns:
        mask_idade_invalida = (df_pad['Idade'] < 18) | (df_pad['Idade'] > 120)
        if mask_idade_invalida.any():
            df_pad.loc[mask_idade_invalida, 'Idade'] = np.nan
            df_pad.loc[mask_idade_invalida, 'Qualidade_Registro'] = df_pad.loc[mask_idade_invalida, 'Qualidade_Registro'].apply(
                lambda x: 'Idade_Invalida' if x == 'OK' else x + '|Idade_Invalida'
            )
            print(f"   ⚠️ Idades inválidas corrigidas para NaN e marcadas: {mask_idade_invalida.sum()} registros")
    
    # --- 7. Padronizar municípios usando cadastro mestre ---
    if cadastro_df is not None and 'Cod_IBGE' in df_pad.columns and 'Cod_IBGE' in cadastro_df.columns:
        municipios_padrao = cadastro_df.set_index('Cod_IBGE')['Municipio_Padrao'].to_dict()
        df_pad['Municipio_Padrao'] = df_pad['Cod_IBGE'].map(municipios_padrao)
        df_pad['Municipio'] = df_pad['Municipio_Padrao'].fillna(df_pad['Municipio'])
        df_pad = df_pad.drop(columns=['Municipio_Padrao'])
        print("   ✅ Municípios padronizados com cadastro mestre")
    
    # --- 8. Padronizar comunidades ---
    if cadastro_df is not None and 'Cod_Com' in df_pad.columns and 'Cod_Comunidade' in cadastro_df.columns:
        comunidades_padrao = cadastro_df.set_index('Cod_Comunidade')['Comunidade_Padrao'].to_dict()
        df_pad['Comunidade_Padrao'] = df_pad['Cod_Com'].map(comunidades_padrao)
        df_pad['Comunidade'] = df_pad['Comunidade_Padrao'].fillna(df_pad['Comunidade'])
        df_pad = df_pad.drop(columns=['Comunidade_Padrao'])
        print("   ✅ Comunidades padronizadas com cadastro mestre")
    
    # --- 9. Campos obrigatórios com nulos (marcar qualidade) ---
    obrigatorios = ['ID', 'UUID', 'Versao_Form', 'Status', 'Municipio', 
                    'Cod_IBGE', 'Comunidade', 'Supervisor', 'Pesquisador',
                    'Consentimento', 'Genero', 'Idade', 'Escolaridade', 'Q1_QV']
    
    for col in obrigatorios:
        if col in df_pad.columns:
            mask_nulo = df_pad[col].isna()
            if mask_nulo.any():
                df_pad.loc[mask_nulo, 'Qualidade_Registro'] = df_pad.loc[mask_nulo, 'Qualidade_Registro'].apply(
                    lambda x: f'Campo_Nulo_{col}' if x == 'OK' else x + f'|Campo_Nulo_{col}'
                )
    
    # --- 10. Status e versão (padronização simples) ---
    if 'Status' in df_pad.columns:
        df_pad['Status'] = df_pad['Status'].str.strip().str.capitalize()
        df_pad.loc[~df_pad['Status'].isin(['Completa', 'Parcial', 'Cancelada']), 'Status'] = 'Parcial'
    
    if 'Versao_Form' in df_pad.columns:
        df_pad['Versao_Form'] = df_pad['Versao_Form'].str.strip()
        df_pad.loc[~df_pad['Versao_Form'].isin(['v2.3', 'v2.4']), 'Versao_Form'] = 'v2.4'
    
    print(f"   ✅ Survey tratado: {len(df_pad)} registros")
    print(f"   📊 Qualidade: {df_pad['Qualidade_Registro'].value_counts().to_dict()}")
    
    return df_pad

def padronizar_evidencias(df, equipe_df):
    """
    Padroniza a base de evidências e aplica critérios do POP
    """
    df_pad = df.copy()
    
    # 1. Status de validação
    if 'Status_Validacao' in df_pad.columns:
        df_pad['Status_Validacao'] = df_pad['Status_Validacao'].str.upper().str.strip()
        validos = ['APROVADO', 'REPROVADO', 'PENDENTE', 'EM_ANALISE']
        df_pad.loc[~df_pad['Status_Validacao'].isin(validos) & df_pad['Status_Validacao'].notna(), 
                   'Status_Validacao'] = 'PENDENTE'
        df_pad.loc[df_pad['Status_Validacao'].isna(), 'Status_Validacao'] = 'PENDENTE'
    
    # 2. Validar pesquisadores com a equipe
    if equipe_df is not None and 'Pesquisador' in df_pad.columns:
        pesquisadores_validos = set(equipe_df[equipe_df['Situação'] == 'ATIVO']['Nome'].tolist())
        df_pad['Pesquisador_Valido'] = df_pad['Pesquisador'].isin(pesquisadores_validos)
    
    # 3. Verificar arquivos obrigatórios (marcar qualidade)
    df_pad['Qualidade_Evidencia'] = 'OK'
    
    if 'Termo_Consentimento' in df_pad.columns and 'Arquivo_Termo' in df_pad.columns:
        mask = (df_pad['Termo_Consentimento'] == 'Sim') & (df_pad['Arquivo_Termo'].isna())
        if mask.any():
            df_pad.loc[mask, 'Qualidade_Evidencia'] = 'TC_Faltando'
    
    if 'Foto_Entrevista' in df_pad.columns and 'Arquivo_Foto' in df_pad.columns:
        mask = (df_pad['Foto_Entrevista'] == 'Sim') & (df_pad['Arquivo_Foto'].isna())
        if mask.any():
            df_pad.loc[mask, 'Qualidade_Evidencia'] = 'Foto_Faltando'
    
    if 'Arquivo_Diario' in df_pad.columns:
        mask = df_pad['Arquivo_Diario'].isna()
        if mask.any():
            df_pad.loc[mask, 'Qualidade_Evidencia'] = 'Diario_Faltando'
    
    # 4. Nomenclatura (apenas marcar, correção será no Script 6)
    def validar_nomenclatura_pop(arquivo, prefixo):
        if pd.isna(arquivo):
            return True
        padrao = rf"^{prefixo}\.[0-9]{{5}}\.(pdf|jpg)$"
        return bool(re.match(padrao, str(arquivo), re.IGNORECASE))
    
    if 'Arquivo_Termo' in df_pad.columns:
        for idx, row in df_pad.iterrows():
            if pd.notna(row['Arquivo_Termo']) and not validar_nomenclatura_pop(row['Arquivo_Termo'], 'TC'):
                if df_pad.loc[idx, 'Qualidade_Evidencia'] == 'OK':
                    df_pad.loc[idx, 'Qualidade_Evidencia'] = 'Nomenclatura_Termo_Incorreta'
                else:
                    df_pad.loc[idx, 'Qualidade_Evidencia'] += '|Nomenclatura_Termo_Incorreta'
    
    if 'Arquivo_Foto' in df_pad.columns:
        for idx, row in df_pad.iterrows():
            if pd.notna(row['Arquivo_Foto']) and not validar_nomenclatura_pop(row['Arquivo_Foto'], 'IMG'):
                if df_pad.loc[idx, 'Qualidade_Evidencia'] == 'OK':
                    df_pad.loc[idx, 'Qualidade_Evidencia'] = 'Nomenclatura_Foto_Incorreta'
                else:
                    df_pad.loc[idx, 'Qualidade_Evidencia'] += '|Nomenclatura_Foto_Incorreta'
    
    if 'Arquivo_Diario' in df_pad.columns:
        for idx, row in df_pad.iterrows():
            if pd.notna(row['Arquivo_Diario']) and not validar_nomenclatura_pop(row['Arquivo_Diario'], 'DC'):
                if df_pad.loc[idx, 'Qualidade_Evidencia'] == 'OK':
                    df_pad.loc[idx, 'Qualidade_Evidencia'] = 'Nomenclatura_Diario_Incorreta'
                else:
                    df_pad.loc[idx, 'Qualidade_Evidencia'] += '|Nomenclatura_Diario_Incorreta'
    
    print(f"   ✅ Evidências tratadas: {len(df_pad)} registros")
    print(f"   📊 Qualidade: {df_pad['Qualidade_Evidencia'].value_counts().to_dict()}")
    
    return df_pad

# ===================================================================
# 2. EXECUÇÃO DO TRATAMENTO
# ===================================================================

print("\n" + "="*70)
print("EXECUTANDO PADRONIZAÇÃO DAS BASES")
print("="*70)

try:
    if 'bases' not in globals() or bases is None:
        print("⚠️ Bases não carregadas. Execute o Script 1 primeiro.")
    else:
        # Padronizar cada base
        bases_padronizadas = {}
        
        # Cadastro
        if 'cadastro' in bases and bases['cadastro'] is not None:
            print("\n📊 Padronizando CADASTRO...")
            bases_padronizadas['cadastro'] = padronizar_cadastro(bases['cadastro'])
        
        # Equipe
        if 'equipe' in bases and bases['equipe'] is not None:
            print("\n📊 Padronizando EQUIPE...")
            bases_padronizadas['equipe'] = padronizar_equipe(bases['equipe'])
        
        # Survey (com cadastro como referência)
        if 'survey' in bases and bases['survey'] is not None:
            print("\n📊 Padronizando SURVEY...")
            cadastro_ref = bases_padronizadas.get('cadastro', None)
            bases_padronizadas['survey'] = padronizar_survey(bases['survey'], cadastro_ref)
        
        # Evidências (com equipe como referência)
        if 'evidencias' in bases and bases['evidencias'] is not None:
            print("\n📊 Padronizando EVIDÊNCIAS...")
            equipe_ref = bases_padronizadas.get('equipe', None)
            bases_padronizadas['evidencias'] = padronizar_evidencias(bases['evidencias'], equipe_ref)
        
        print("\n" + "="*70)
        print("✅ TRATAMENTO CONCLUÍDO!")
        print("="*70)
        print("\n📋 Resumo das bases tratadas:")
        for nome, df in bases_padronizadas.items():
            print(f"   - {nome}: {len(df)} registros, {len(df.columns)} colunas")
        
        # Salvar uma cópia para uso posterior (opcional)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        for nome, df in bases_padronizadas.items():
            df.to_csv(f"base_tratada_{nome}_{timestamp}.csv", index=False, encoding='utf-8-sig')
        print(f"\n📁 Cópias das bases tratadas salvas como 'base_tratada_*.csv'")

except NameError:
    print("⚠️ Variável 'bases' não definida. Execute o Script 1 primeiro.")
except Exception as e:
    print(f"❌ Erro durante o tratamento: {e}")
    import traceback
    traceback.print_exc()

INICIANDO TRATAMENTO E PADRONIZAÇÃO DOS DADOS

EXECUTANDO PADRONIZAÇÃO DAS BASES

✅ TRATAMENTO CONCLUÍDO!

📋 Resumo das bases tratadas:

📁 Cópias das bases tratadas salvas como 'base_tratada_*.csv'


Script_5_Exportacao_Final

In [6]:
# ===================================================================
# EXPORTAÇÃO DOS DADOS TRATADOS
# ===================================================================

import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
import sys

def gerar_estatisticas_completas(df, nome_base):
    """
    Gera estatísticas detalhadas para uma base de dados
    """
    estatisticas = {
        'nome_base': nome_base,
        'timestamp': datetime.now().isoformat(),
        'visao_geral': {
            'total_registros': len(df),
            'total_colunas': len(df.columns),
            'memoria_uso': f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
            'colunas_numericas': len(df.select_dtypes(include=[np.number]).columns),
            'colunas_categoricas': len(df.select_dtypes(include=['object', 'category']).columns),
            'colunas_datetime': len(df.select_dtypes(include=['datetime64']).columns)
        },
        'qualidade_dados': {
            'duplicatas_exatas': int(df.duplicated().sum()),
            'percentual_duplicatas': f"{(df.duplicated().sum() / len(df) * 100):.2f}%" if len(df) > 0 else "0%",
            'total_nulos': int(df.isnull().sum().sum()),
            'percentual_nulos': f"{(df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100):.2f}%" if len(df) > 0 else "0%"
        },
        'colunas': {}
    }
    
    # Estatísticas por coluna
    for col in df.columns:
        col_info = {
            'tipo': str(df[col].dtype),
            'nulos': int(df[col].isnull().sum()),
            'percentual_nulos': f"{(df[col].isnull().sum() / len(df) * 100):.2f}%" if len(df) > 0 else "0%",
            'unicos': int(df[col].nunique()) if len(df) > 0 else 0
        }
        
        # Estatísticas específicas por tipo
        if pd.api.types.is_numeric_dtype(df[col]):
            col_info.update({
                'min': float(df[col].min()) if not df[col].isnull().all() else None,
                'max': float(df[col].max()) if not df[col].isnull().all() else None,
                'mean': float(df[col].mean()) if not df[col].isnull().all() else None,
                'median': float(df[col].median()) if not df[col].isnull().all() else None,
                'std': float(df[col].std()) if not df[col].isnull().all() else None
            })
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            if not df[col].isnull().all():
                col_info.update({
                    'min': df[col].min().isoformat() if pd.notna(df[col].min()) else None,
                    'max': df[col].max().isoformat() if pd.notna(df[col].max()) else None
                })
        else:
            # Colunas categóricas/texto
            if df[col].nunique() <= 20 and len(df) > 0:  # Mostrar valores frequentes
                value_counts = df[col].value_counts().head(10)
                col_info['valores_frequentes'] = {
                    str(k): int(v) for k, v in value_counts.items()
                }
        
        estatisticas['colunas'][col] = col_info
    
    return estatisticas

def exportar_bases_completo(bases_dict, pasta_saida, prefixo='tratado'):
    """
    Exporta todas as bases tratadas em CSV, Excel e Parquet com estatísticas
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Criar pasta para a execução
    pasta_execucao = os.path.join(pasta_saida, f'execucao_{timestamp}')
    os.makedirs(pasta_execucao, exist_ok=True)
    
    # Criar subpasta para estatísticas
    pasta_estatisticas = os.path.join(pasta_execucao, 'estatisticas')
    os.makedirs(pasta_estatisticas, exist_ok=True)
    
    relatorio = []
    estatisticas_gerais = {}
    
    print("\n" + "="*70)
    print("EXPORTANDO BASES TRATADAS")
    print("="*70)
    
    for nome, df in bases_dict.items():
        if df is None or len(df) == 0:
            print(f"⚠️ {nome}: Base vazia ou None - pulando...")
            continue
        
        print(f"\n📊 Processando: {nome}")
        print(f"   Registros: {len(df):,} | Colunas: {len(df.columns)}")
        
        # 1. Gerar estatísticas
        print(f"   Gerando estatísticas...")
        estatisticas = gerar_estatisticas_completas(df, nome)
        estatisticas_gerais[nome] = estatisticas
        
        # Salvar estatísticas em JSON
        json_path = os.path.join(pasta_estatisticas, f'estatisticas_{nome}_{timestamp}.json')
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(estatisticas, f, ensure_ascii=False, indent=2, default=str)
        relatorio.append(f"📊 Estatísticas JSON: {json_path}")
        
        # Salvar estatísticas em TXT (formato legível)
        txt_path = os.path.join(pasta_estatisticas, f'estatisticas_{nome}_{timestamp}.txt')
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write("="*80 + "\n")
            f.write(f"RELATÓRIO DE ESTATÍSTICAS - {nome}\n")
            f.write("="*80 + "\n\n")
            
            # Visão geral
            f.write("VISÃO GERAL\n")
            f.write("-"*40 + "\n")
            for key, value in estatisticas['visao_geral'].items():
                f.write(f"{key:25}: {value}\n")
            
            f.write("\nQUALIDADE DOS DADOS\n")
            f.write("-"*40 + "\n")
            for key, value in estatisticas['qualidade_dados'].items():
                f.write(f"{key:25}: {value}\n")
            
            f.write("\nDETALHAMENTO POR COLUNA\n")
            f.write("-"*40 + "\n")
            for col, info in estatisticas['colunas'].items():
                f.write(f"\n{col}:\n")
                for key, value in info.items():
                    if key != 'valores_frequentes':
                        f.write(f"  {key:20}: {value}\n")
                    else:
                        f.write(f"  {key:20}:\n")
                        for val, count in value.items():
                            f.write(f"    {val:20}: {count}\n")
        
        relatorio.append(f"📊 Estatísticas TXT: {txt_path}")
        
        # 2. Salvar como CSV
        csv_path = os.path.join(pasta_execucao, f'{prefixo}_{nome}_{timestamp}.csv')
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        relatorio.append(f"✅ CSV: {csv_path}")
        
        # 3. Salvar como Excel
        try:
            xlsx_path = os.path.join(pasta_execucao, f'{prefixo}_{nome}_{timestamp}.xlsx')
            with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
                df.to_excel(writer, sheet_name='Dados', index=False)
                
                # Adicionar sheet com estatísticas
                stats_df = pd.DataFrame(estatisticas['colunas']).T
                stats_df.to_excel(writer, sheet_name='Estatisticas')
            relatorio.append(f"✅ Excel: {xlsx_path}")
        except Exception as e:
            relatorio.append(f"⚠️ Erro ao salvar Excel: {e}")
        
        # 4. Salvar como Parquet (COMPACTADO e RÁPIDO)
        try:
            parquet_path = os.path.join(pasta_execucao, f'{prefixo}_{nome}_{timestamp}.parquet')
            df.to_parquet(parquet_path, index=False, compression='snappy')
            # Obter tamanho do arquivo
            tamanho_bytes = os.path.getsize(parquet_path)
            tamanho_mb = tamanho_bytes / (1024 * 1024)
            relatorio.append(f"✅ Parquet: {parquet_path} ({tamanho_mb:.2f} MB)")
        except Exception as e:
            relatorio.append(f"⚠️ Erro ao salvar Parquet: {e}")
        
        # 5. Criar resumo da base
        resumo = {
            'nome': nome,
            'registros': len(df),
            'colunas': len(df.columns),
            'duplicatas': df.duplicated().sum(),
            'nulos_totais': df.isnull().sum().sum(),
            'memoria': df.memory_usage(deep=True).sum()
        }
        relatorio.append(f"📊 Resumo: {resumo}")
        relatorio.append("-"*50)
        
        print(f"   ✅ Exportado com sucesso!")
    
    # ===================================================================
    # RELATÓRIO GERAL CONSOLIDADO
    # ===================================================================
    
    print("\n" + "="*70)
    print("GERANDO RELATÓRIO GERAL CONSOLIDADO")
    print("="*70)
    
    # 1. Salvar relatório consolidado das estatísticas
    relatorio_consolidado = {
        'timestamp': timestamp,
        'bases_processadas': len(estatisticas_gerais),
        'resumo_geral': {}
    }
    
    for nome, stats in estatisticas_gerais.items():
        relatorio_consolidado['resumo_geral'][nome] = {
            'registros': stats['visao_geral']['total_registros'],
            'colunas': stats['visao_geral']['total_colunas'],
            'duplicatas': stats['qualidade_dados']['duplicatas_exatas'],
            'nulos': stats['qualidade_dados']['total_nulos']
        }
    
    # Salvar consolidado em JSON
    consolidado_json = os.path.join(pasta_execucao, f'relatorio_consolidado_{timestamp}.json')
    with open(consolidado_json, 'w', encoding='utf-8') as f:
        json.dump(relatorio_consolidado, f, ensure_ascii=False, indent=2, default=str)
    relatorio.append(f"📊 Relatório consolidado JSON: {consolidado_json}")
    
    # 2. Salvar relatório de exportação em TXT
    relatorio_path = os.path.join(pasta_execucao, f'relatorio_exportacao_{timestamp}.txt')
    with open(relatorio_path, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("RELATÓRIO DE EXPORTAÇÃO - PIPELINE DE DADOS\n")
        f.write("="*80 + "\n\n")
        f.write(f"Data de execução: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"Bases processadas: {len(estatisticas_gerais)}\n\n")
        f.write("-"*80 + "\n\n")
        f.write("\n".join(relatorio))
        
        # Adicionar resumo consolidado
        f.write("\n\n" + "="*80 + "\n")
        f.write("RESUMO CONSOLIDADO\n")
        f.write("="*80 + "\n\n")
        for nome, resumo in relatorio_consolidado['resumo_geral'].items():
            f.write(f"{nome}:\n")
            for key, value in resumo.items():
                f.write(f"  {key:15}: {value:,}\n")
            f.write("\n")
    
    # 3. Salvar sumário executivo em Markdown (para documentação)
    md_path = os.path.join(pasta_execucao, f'resumo_executivo_{timestamp}.md')
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(f"# Relatório de Exportação - {timestamp}\n\n")
        f.write(f"**Data de execução:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("## Resumo das Bases Processadas\n\n")
        f.write("| Base | Registros | Colunas | Duplicatas | Nulos |\n")
        f.write("|------|-----------|---------|------------|-------|\n")
        for nome, resumo in relatorio_consolidado['resumo_geral'].items():
            f.write(f"| {nome} | {resumo['registros']:,} | {resumo['colunas']} | {resumo['duplicatas']:,} | {resumo['nulos']:,} |\n")
        
        f.write("\n## Arquivos Gerados\n\n")
        for line in relatorio:
            if line.startswith('✅') or line.startswith('📊'):
                f.write(f"- {line}\n")
    
    print(f"\n📄 Relatório de exportação: {relatorio_path}")
    print(f"📊 Relatório consolidado: {consolidado_json}")
    print(f"📝 Sumário executivo: {md_path}")
    
    return pasta_execucao

# ===================================================================
# EXECUTAR EXPORTAÇÃO
# ===================================================================

# Verificar se as bases padronizadas existem
try:
    if 'bases_padronizadas' in globals() and bases_padronizadas is not None:
        print("\n" + "="*70)
        print("INICIANDO EXPORTAÇÃO COMPLETA")
        print("="*70)
        
        # Definir pasta de saída
        PASTA_OUTPUT = 'output_tratado'
        os.makedirs(PASTA_OUTPUT, exist_ok=True)
        
        # Executar exportação
        pasta_exportacao = exportar_bases_completo(
            bases_padronizadas, 
            PASTA_OUTPUT, 
            prefixo='tratado'
        )
        
        print("\n" + "="*70)
        print("✅ EXPORTAÇÃO CONCLUÍDA COM SUCESSO!")
        print("="*70)
        print(f"\n📁 Pasta de saída: {pasta_exportacao}")
        print("\n📋 Arquivos gerados:")
        print("   - CSV: Dados em formato universal")
        print("   - Excel: Dados + estatísticas em abas separadas")
        print("   - Parquet: Dados compactados e de alta performance")
        print("   - Estatísticas: JSON e TXT com métricas detalhadas")
        print("   - Relatórios: Consolidado e executivo")
        
    else:
        print("⚠️ Bases padronizadas não encontradas. Execute o Script anterior primeiro.")
        print("   Ou verifique se a variável 'bases_padronizadas' existe.")
        
except NameError:
    print("⚠️ Variável 'bases_padronizadas' não definida.")
    print("   Execute o Script 4 (Tratamento e Padronização) primeiro.")
except Exception as e:
    print(f"❌ Erro durante a exportação: {e}")
    import traceback
    traceback.print_exc()
    


INICIANDO EXPORTAÇÃO COMPLETA

EXPORTANDO BASES TRATADAS

GERANDO RELATÓRIO GERAL CONSOLIDADO

📄 Relatório de exportação: output_tratado\execucao_20260723_035815\relatorio_exportacao_20260723_035815.txt
📊 Relatório consolidado: output_tratado\execucao_20260723_035815\relatorio_consolidado_20260723_035815.json
📝 Sumário executivo: output_tratado\execucao_20260723_035815\resumo_executivo_20260723_035815.md

✅ EXPORTAÇÃO CONCLUÍDA COM SUCESSO!

📁 Pasta de saída: output_tratado\execucao_20260723_035815

📋 Arquivos gerados:
   - CSV: Dados em formato universal
   - Excel: Dados + estatísticas em abas separadas
   - Parquet: Dados compactados e de alta performance
   - Estatísticas: JSON e TXT com métricas detalhadas
   - Relatórios: Consolidado e executivo


Script_6_Renomeacao_Arquivos

In [7]:
# ===================================================================
# RENOMEAÇÃO AUTOMÁTICA DE ARQUIVOS
# ===================================================================
# Este script renomeia os arquivos de TC, IMG e DC automaticamente para o padrão 
# resolvendo 743 ocorrências de nomenclatura:
#         
#         TC.#####.pdf | IMG.#####.jpg | DC.#####.pdf
#
# ORIENTAÇÃO DE USO:
# 1) Localize e edite a variável PASTA_RAIZ_ARQUIVOS com o caminho real onde os arquivos estão armazenados. 
#         Exemplo: "C:/Projeto/ArquivosCampo"
# 2) Execute este script que irá fazer a seguinte pergunta: 
#        "Digite o modo de operação (1=Simulação, 2=Execução):"
#    em que: 
#     1 - SIMULAÇÃO - apenas verifica os arquivos existentes na pasta sem renomeiar;
#     2 - EXECUÇÃO - renomeia os arquivos existentemente e gera relatório final;

# 3) Digite 1 que fará a Simulação.
# 4) Analise o relatório gerado para ver quais arquivos serão renomeados, quais não foram encontrados e afins.
#    Localizer pelo nome: relatorio_renomeacao_YYYYMMDD_HHMMSS.csv 
# 5) Se estiver satisfeito(a) com a simulação, execute novamente o script e digite 2 para fazer a Execução 
#    e confirme digitando SIM quando for solicitado.
# 6) Ao final, renomeará os arquivos fisicamente e gerará um novo relatório.
# ===================================================================

import pandas as pd
import os
import re
import shutil
from datetime import datetime
from pathlib import Path

print("="*70)
print("RENOMEAÇÃO AUTOMÁTICA DE ARQUIVOS")
print("="*70)

# ===================================================================
# CONFIGURAÇÃO - AJUSTE ESTE CAMINHO!
# ===================================================================
# Defina o caminho da pasta raiz onde os arquivos estão armazenados
PASTA_RAIZ_ARQUIVOS = "caminho/para/pasta/de/arquivos"  # <-- ALTERE AQUI!

# Se não souber o caminho exato, use esta opção para descobrir:
# PASTA_RAIZ_ARQUIVOS = input("Digite o caminho da pasta com os arquivos: ")

# ===================================================================
# 1. CARREGAR A BASE DE EVIDÊNCIAS TRATADA
# ===================================================================

def carregar_base_evidencias(caminho_csv=None):
    """
    Carrega a base de evidências tratada.
    Se não informar caminho, procura pelo arquivo mais recente.
    """
    if caminho_csv and os.path.exists(caminho_csv):
        return pd.read_csv(caminho_csv)
    
    # Procurar arquivo mais recente
    import glob
    arquivos = glob.glob('base_tratada_evidencias_*.csv')
    if arquivos:
        ultimo = sorted(arquivos)[-1]
        print(f"📂 Carregando: {ultimo}")
        return pd.read_csv(ultimo)
    
    # Tentar carregar do output
    arquivos = glob.glob('output_tratado/**/tratado_evidencias_*.csv', recursive=True)
    if arquivos:
        ultimo = sorted(arquivos)[-1]
        print(f"📂 Carregando: {ultimo}")
        return pd.read_csv(ultimo)
    
    raise FileNotFoundError("❌ Nenhum arquivo de evidências encontrado!")

# ===================================================================
# 2. FUNÇÕES DE RENOMEAÇÃO
# ===================================================================

def validar_nomenclatura_pop(arquivo, prefixo):
    """Verifica se o nome segue o padrão POP"""
    if pd.isna(arquivo):
        return False
    padrao = rf"^{prefixo}\.[0-9]{{5}}\.(pdf|jpg)$"
    return bool(re.match(padrao, str(arquivo), re.IGNORECASE))

def extrair_id_do_arquivo(arquivo, prefixo):
    """Tenta extrair o ID do nome do arquivo"""
    if pd.isna(arquivo):
        return None
    # Tenta padrão TC_1234.pdf ou TC.1234.pdf
    match = re.search(rf"{prefixo}[._](\d+)\.", str(arquivo), re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

def renomear_arquivo(caminho_antigo, caminho_novo, dry_run=True):
    """
    Renomeia um arquivo.
    Se dry_run=True, apenas simula (não altera nada).
    """
    if not os.path.exists(caminho_antigo):
        return 'nao_encontrado', f"Arquivo não encontrado: {caminho_antigo}"
    
    if os.path.exists(caminho_novo):
        return 'destino_existe', f"Destino já existe: {caminho_novo}"
    
    if dry_run:
        return 'simulado', f"Simulação: {os.path.basename(caminho_antigo)} -> {os.path.basename(caminho_novo)}"
    
    try:
        os.rename(caminho_antigo, caminho_novo)
        return 'sucesso', f"Renomeado: {os.path.basename(caminho_antigo)} -> {os.path.basename(caminho_novo)}"
    except Exception as e:
        return 'erro', f"Erro: {e}"

def executar_renomeacao(df, pasta_raiz, dry_run=True):
    """
    Executa a renomeação de todos os arquivos com nomenclatura incorreta.
    """
    resultados = {
        'Termo': {'sucesso': 0, 'simulado': 0, 'nao_encontrado': 0, 'destino_existe': 0, 'erro': 0, 'pular': 0},
        'Foto': {'sucesso': 0, 'simulado': 0, 'nao_encontrado': 0, 'destino_existe': 0, 'erro': 0, 'pular': 0},
        'Diario': {'sucesso': 0, 'simulado': 0, 'nao_encontrado': 0, 'destino_existe': 0, 'erro': 0, 'pular': 0},
        'detalhes': []
    }
    
    # Mapeamento de colunas
    colunas = {
        'Termo': ('Arquivo_Termo', 'TC', '.pdf'),
        'Foto': ('Arquivo_Foto', 'IMG', '.jpg'),
        'Diario': ('Arquivo_Diario', 'DC', '.pdf')
    }
    
    for idx, row in df.iterrows():
        id_entrevista = row.get('ID_Entrevista', row.get('ID'))
        if pd.isna(id_entrevista):
            continue
        
        for tipo, (col_arquivo, prefixo, extensao) in colunas.items():
            if col_arquivo not in row:
                continue
            
            arquivo_atual = row[col_arquivo]
            if pd.isna(arquivo_atual):
                continue
            
            # Se já estiver no padrão correto, pular
            if validar_nomenclatura_pop(arquivo_atual, prefixo):
                resultados[tipo]['pular'] += 1
                continue
            
            # Nome correto
            nome_correto = f"{prefixo}.{int(id_entrevista):05d}{extensao}"
            
            # Tentar encontrar o arquivo
            # Buscar em subpastas recursivamente
            arquivos_encontrados = list(Path(pasta_raiz).rglob(arquivo_atual))
            if not arquivos_encontrados:
                # Tentar buscar pelo nome sem caminho
                arquivos_encontrados = list(Path(pasta_raiz).rglob(f"*{os.path.basename(arquivo_atual)}"))
            
            if not arquivos_encontrados:
                resultados[tipo]['nao_encontrado'] += 1
                resultados['detalhes'].append({
                    'ID': id_entrevista,
                    'Tipo': tipo,
                    'Arquivo_Atual': arquivo_atual,
                    'Status': 'nao_encontrado',
                    'Mensagem': f"Arquivo não encontrado na pasta: {arquivo_atual}"
                })
                continue
            
            # Pega o primeiro encontrado
            caminho_antigo = arquivos_encontrados[0]
            caminho_novo = caminho_antigo.parent / nome_correto
            
            status, mensagem = renomear_arquivo(str(caminho_antigo), str(caminho_novo), dry_run)
            resultados[tipo][status] += 1
            resultados['detalhes'].append({
                'ID': id_entrevista,
                'Tipo': tipo,
                'Arquivo_Atual': arquivo_atual,
                'Arquivo_Correto': nome_correto,
                'Status': status,
                'Mensagem': mensagem
            })
    
    return resultados

# ===================================================================
# 3. EXECUÇÃO PRINCIPAL
# ===================================================================

try:
    # Carregar base de evidências
    df_evidencias = carregar_base_evidencias()
    print(f"✅ Base carregada: {len(df_evidencias)} registros")
    
    # Verificar se a pasta existe
    if not os.path.exists(PASTA_RAIZ_ARQUIVOS):
        print(f"\n⚠️ ATENÇÃO: A pasta '{PASTA_RAIZ_ARQUIVOS}' não existe!")
        print("   Por favor, ajuste a variável PASTA_RAIZ_ARQUIVOS no início do script.")
        print("   Exemplo: PASTA_RAIZ_ARQUIVOS = 'C:/MeusDocumentos/ArquivosCampo'")
        PASTA_RAIZ_ARQUIVOS = input("\nDigite o caminho correto da pasta: ")
        if not os.path.exists(PASTA_RAIZ_ARQUIVOS):
            print("❌ Pasta ainda não encontrada. Encerrando.")
            exit()
    
    # Modo de execução
    print(f"\n📁 Pasta de arquivos: {PASTA_RAIZ_ARQUIVOS}")
    print("\nModos de execução:")
    print("   1 - SIMULAÇÃO (apenas verifica, não renomeia)")
    print("   2 - EXECUTAR (renomeia os arquivos)")
    modo = input("\nEscolha o modo (1 ou 2): ").strip()
    
    dry_run = modo == '1'
    if dry_run:
        print("\n🔍 Modo SIMULAÇÃO ativado. Nenhum arquivo será alterado.")
    else:
        print("\n⚠️ Modo EXECUÇÃO ativado. Arquivos serão RENOMEADOS!")
        confirmar = input("Tem certeza? Digite 'SIM' para continuar: ").strip().upper()
        if confirmar != 'SIM':
            print("❌ Operação cancelada.")
            exit()
    
    # Executar renomeação
    print("\n🔄 Processando renomeação...")
    resultados = executar_renomeacao(df_evidencias, PASTA_RAIZ_ARQUIVOS, dry_run)
    
    # Resumo
    print("\n" + "="*70)
    print("RESUMO DA RENOMEAÇÃO")
    print("="*70)
    
    for tipo, stats in resultados.items():
        if tipo == 'detalhes':
            continue
        print(f"\n{tipo}:")
        print(f"   ✅ Sucesso: {stats['sucesso']}")
        print(f"   🔍 Simulado: {stats['simulado']}")
        print(f"   ⏭️  Pulados (já corretos): {stats['pular']}")
        print(f"   ❌ Não encontrados: {stats['nao_encontrado']}")
        print(f"   ⚠️ Destino já existe: {stats['destino_existe']}")
        print(f"   🚫 Erros: {stats['erro']}")
    
    total_processados = sum(
        resultados[tipo]['sucesso'] + resultados[tipo]['simulado'] + 
        resultados[tipo]['nao_encontrado'] + resultados[tipo]['destino_existe'] + 
        resultados[tipo]['erro']
        for tipo in ['Termo', 'Foto', 'Diario']
    )
    total_pulados = sum(resultados[tipo]['pular'] for tipo in ['Termo', 'Foto', 'Diario'])
    print(f"\n📊 Total de dados processados: {total_processados}")
    print(f"📊 Total de dados anteriormente corretos): {total_pulados}")
    
    # Salvar relatório
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    relatorio_df = pd.DataFrame(resultados['detalhes'])
    relatorio_path = f"relatorio_renomeacao_{timestamp}.csv"
    relatorio_df.to_csv(relatorio_path, index=False, encoding='utf-8-sig')
    print(f"\n📄 Relatório detalhado salvo em: {relatorio_path}")
    
    if dry_run:
        print("\n💡 Esta foi uma SIMULAÇÃO. Para renomear de fato, execute novamente digitando 2 para realizar a EXECUÇÃO.")
    else:
        print("\n✅ Renomeação concluída!")

except FileNotFoundError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Erro: {e}")
    import traceback
    traceback.print_exc()

RENOMEAÇÃO AUTOMÁTICA DE ARQUIVOS
❌ ❌ Nenhum arquivo de evidências encontrado!
